# OGR – Dataworks Transformation Approach (Final)

Updates: "Dataworks" is the internal RTLH Python framework that wraps **Databricks Asset Bundles (DAB) + Flyway + reusable ingestion/transformation notebooks** (FullMerge, DeltaLoadUsingCT, PersistedCDF, schema-drift detection, Key Vault integration). OGR builds a Dataworks **project** per dataset; ingestion is owned by Data Integration.

---

## 1. Current PRiME / Synapse DW (recap)

Layered Synapse SQL solution under 1. DEV:

- **UDL (mocked HSTG_V / DAL_V)** → **HLP_DEV** (per-source `vw_hstg` / `vw_dim` / `vw_fact` + custom CDC via `rep_*` + `vw_cdc_*` + `cdc.*`) → **PRES_DEV** (OCDM-aligned `dim_*`, `fact_*`, `m_fact_*`) → **PRES_<DATASET>_DEV** (HME / RAIL / EEDRILL / EEDRILLCOST / AM / PROCESS / PROCESSASSURANCE / GLOBAL_METRICS star schemas) → **DS Tabular models / GMR**.
- Refresh = Synapse Data Flows driven by Excel-backed metadata in MiDAS.
- Deploy = DACPAC compare → SQL script → manual SSMS run, with folder staging via Stage SQL Objects.ps1.

Pain to remove: Synapse Dedicated Pool cost, DACPAC limitations, hand-rolled CDC, identity-SK plumbing (RemoveIdentityColumn.sql), `1. DEV / 2. TEST / 3. PROD` folder staging, MiDAS-driven dataflow metadata, no sandboxing per developer.

---

## 2. Source contract from Data Integration (out of OGR scope)

- Catalog: **`_ogr_azr_syd`**, schema per source system; same `DAL_V.*` / `HSTG_V.*` entities used today.
- Each table is **Delta with CDF enabled** plus two metadata columns:
  - `rtlh_row_hash` — hash of business columns.
  - `rtlh_ingestion_time` — last ingest timestamp.
- Refresh = MERGE keyed on `rtlh_row_hash`: unchanged row → no-op; changed row → old hash deleted + new hash inserted; dropped source row → deleted; new row → inserted.

Implications for OGR:
1. **No bespoke CDC.** The `HLP_DEV.rep_*` / `vw_cdc_*` / `cdc.*` stack is replaced by reading `table_changes('_ogr_azr_syd.<source>.<tbl>', start_version)`.
2. CDF for an update arrives as a `(delete, insert)` pair at the same `_commit_version` — never `update_preimage` / `update_postimage`.
3. **Join downstream on business keys**, not `rtlh_row_hash` (the hash changes when the row changes).
4. Hard deletes propagate, so OCDM dim SCD logic must close current rows / set `is_deleted` (replaces `OMD_DELETED_RECORD_INDICATOR`).
5. The Dataworks `PersistedCDF` reusable notebook is the canonical way to consume this contract.
6. The UDL mocking workflow (`[HLP_DEV].[sp_UDLVWDesc]`) is obsolete.

---

## 3. Target architecture on Databricks + Dataworks

```mermaid
flowchart LR
    subgraph DI["Data Integration (out of OGR scope)"]
        SRC[Source systems]
        ING["_ogr_azr_syd.&lt;source&gt;.*<br/>Delta + CDF<br/>rtlh_row_hash / rtlh_ingestion_time"]
        SRC --> ING
    end

    subgraph OGR["OGR Dataworks projects (one per dataset)"]
      direction TB
      subgraph HLP["prime_silver.hlp_&lt;source&gt;"]
        H1[Per-source shaping<br/>Dataworks reusable notebooks<br/>PersistedCDF / DeltaLoadUsingCT / FullMerge]
      end
      subgraph OCDM["prime_silver.ocdm (OCDM-aligned)"]
        O1[dim_* SCD1/SCD2<br/>hash SK = xxhash64 NK]
        O2[fact_* / m_fact_*<br/>partition by month]
      end
      subgraph GOLD["prime_gold.&lt;dataset&gt; (Star schema)"]
        G1[dim_* / fact_* per OGR dataset]
      end
    end

    subgraph Sem["Semantic layer"]
        M1[GMR metrics / Tabular / Power BI on Databricks SQL Warehouse]
    end

    ING --> HLP --> OCDM --> GOLD --> Sem

    Erwin[(OCDM<br/>Erwin)] -.contract.-> OCDM
    Flyway[Flyway CLI<br/>schemas / tables / views] -.deploys.-> HLP
    Flyway -.deploys.-> OCDM
    Flyway -.deploys.-> GOLD
    DAB[Databricks Asset Bundle<br/>jobs / notebooks / pipelines] -.orchestrates.-> HLP
    DAB -.orchestrates.-> OCDM
    DAB -.orchestrates.-> GOLD
    KV[Azure Key Vault] -.auth.-> DAB
```

| Target | Replaces | Owner / Tooling |
|---|---|---|
| `_ogr_azr_syd.<source>.*` | `HSTG_V.*` / `DAL_V.*` + UDL mocks | Data Integration |
| `prime_silver.hlp_<source>.*` | `HLP_DEV.vw_hstg/vw_dim/vw_fact` + `rep_*`/`vw_cdc_*`/`cdc.*` | OGR Dataworks (notebooks + Flyway DDL) |
| `prime_silver.ocdm.dim_*/fact_*` | `PRES_DEV.dim_*/fact_*/m_fact_*` | OGR Dataworks (Erwin contract) |
| `prime_gold.<dataset>.*` | `PRES_<DATASET>_DEV.*` | OGR Dataworks |
| GMR / Tabular | DS Tabular models in DS/ | Re-pointed to Databricks SQL Warehouse |

---

## 4. Standard Dataworks project layout (per OGR dataset)

A Dataworks project enforces a structured folder layout, multi-env DAB targets, sandboxes, and Flyway migrations. Recommended layout per dataset (e.g. `ogr_hme`):

```
ogr_hme/
├── databricks.yml                  # DAB root (targets: dev, test, prod, sandbox-<dev>)
├── resources/                      # DAB jobs, schedules, clusters/SQL warehouses
│   └── ogr_hme.job.yml
├── notebooks/
│   ├── hlp/                        # per-source shaping (uses Dataworks reusable notebooks)
│   ├── ocdm/                       # OCDM dims/facts
│   └── gold/                       # star-schema models
├── flyway/                         # SQL DDL migrations, versioned
│   ├── V001__create_schemas.sql
│   ├── V002__hlp_hme_tables.sql
│   ├── V010__ocdm_hme_dims.sql
│   └── R__views_gold_hme.sql       # repeatable views
├── config/
│   ├── ingestion.yml               # metadata-driven HLP table list
│   └── ocdm_mapping.yml            # OCDM/Erwin-derived mapping
├── tests/                          # row-count / sum / reconciliation tests
└── pyproject.toml / requirements   # pins Dataworks package version (Artifactory)
```

This replaces today's `1. DEV / 2. TEST / 3. PROD` folder duplication and the Stage SQL Objects.ps1 staging step — environments are DAB targets, not folders.

### 4.1 Use of the three Dataworks components

| Component | Used for in OGR |
|---|---|
| **DAB CLI** | Deploys the dataset job graph (HLP → OCDM → Gold), notebooks, Python wheels, schedules; supports per-developer sandbox targets. Replaces MiDAS-driven dataflow metadata. |
| **Flyway CLI** | Versions and deploys all DDL: `prime_silver.hlp_*`, `prime_silver.ocdm.*`, `prime_gold.<ds>.*` tables/views. Replaces DACPAC + manual SSMS deployment. |
| **Reusable notebooks** | `PersistedCDF` for HLP incremental; `FullMerge` / `DeltaLoadUsingCT` where applicable; schema-drift detection for source evolution; Key Vault integration. |

---

## 5. Patterns per layer

### 5.1 HLP — per-source shaping (incremental from CDF)

- Driven by **`PersistedCDF`** reusable notebook, parameterised by `(source_table, target_table, business_key, transform_sql)`.
- Bookmark = last processed `_commit_version` per (source, target), persisted in a Dataworks state table.
- Treat `(delete, insert)` at same `_commit_version` as logical update — handled naturally by `MERGE` on the deterministic hash SK.
- One HLP model per current `HLP_DEV` view → enables 1:1 reconciliation.
- Schema drift: handled automatically by the Dataworks framework (no manual `[HLP_DEV].[sp_UDLVWDesc]` re-mocking).

### 5.2 OCDM — cross-source conformance

- One Dataworks notebook per Erwin entity → `prime_silver.ocdm.*`.
- Surrogate keys via `xxhash64(concat_ws('|', <natural keys>))` — deterministic, idempotent, no IDENTITY columns.
- SCD2 via Dataworks `FullMerge` macro pattern on attributes flagged historised in Erwin; default Type-1.
- Hard deletes propagated as `is_deleted = true` + close current row.
- Facts partitioned by event-month; `m_fact` = partition filter (`>= add_months(current_date(), -6)`) + `OPTIMIZE … ZORDER BY` on hot dim FKs. Replaces today's Load fact partition.sql procedural pattern.

### 5.3 Gold — per-dataset star schema

- One Dataworks notebook (or Flyway-managed view) per current `PRES_<DATASET>_DEV` object.
- Strip technical columns; implement role-playing splits as separate gold objects (e.g. `fact_HMEBaseTimeUsageEvent`, `fact_HMEGroupTimeUsageEvent`, `fact_HMERecordTimeUsageEvent` from one OCDM fact).
- Keep physical names → mechanical Power BI / GMR re-binding.

### 5.4 Orchestration, environments, security

- **DAB job per dataset** wires HLP → OCDM → Gold tasks and schedules.
- **Targets**: `dev`, `test`, `prod`, plus `sandbox-<developer>` for isolated work. Replaces the per-environment SQL folders.
- **Auth**: Azure Key Vault via Dataworks integration — replaces the manual sqlcmd / SSMS credential pattern.
- **CI/CD**: pipelines in CICD/ re-templated to run `dataworks build` → `flyway migrate` → `databricks bundle deploy --target <env>`. Replaces the DACPAC-compare + GO-stripping workaround in PRIME.CICD.DW-uses local server.yml.

---

## 6. Relationship between artefacts

| Artefact | Owns | Drives |
|---|---|---|
| **Dataworks project (DAB + Flyway + notebooks)** | All OGR code, DDL, and orchestration. | Build order, tests, multi-env deploy, sandboxes. |
| **OCDM model in Erwin** | Logical contract for `prime_silver.ocdm.*`. | Column names, datatypes, SCD type, conformed keys; drives the Flyway DDL and OCDM notebooks. |
| **Star schema (`prime_gold.<ds>`)** | Physical analytical model per OGR dataset. | Query shape, role-playing dims, partitioning. |
| **GMR metrics** | Business KPI definitions in the semantic layer. | Tabular measures + parity tests. |

---

## 7. Reimplementation guidelines

1. **Out of scope for OGR**: UDL mocked views, `[HLP_DEV].[sp_UDLVWDesc]`, all `rep_*` / `vw_cdc_*` / `cdc.*` objects, and the `1. DEV / 2. TEST / 3. PROD` staging mechanism. Replaced by Data Integration's CDF contract + DAB targets.
2. **Per legacy SQL file** classify into HLP / OCDM / gold and create the matching Dataworks asset (notebook + Flyway DDL).
3. **CDF consumption**: use `PersistedCDF`; persist `_commit_version` bookmark; treat `(delete, insert)` pairs as logical update; never join on `rtlh_row_hash`.
4. **Surrogate keys**: deterministic `xxhash64`. Removes RemoveIdentityColumn.sql.
5. **SCD**: default Type-1; Type-2 only where Erwin marks the attribute historised.
6. **Deletes**: propagate as `is_deleted = true` + close current SCD2 row.
7. **Naming**: keep physical names (`fact_HMEGroupTimeUsageEvent`, etc.) in gold to minimise GMR / Power BI churn.
8. **Sandbox first**: developers work in `sandbox-<name>` DAB targets writing to dev catalogs prefixed by sandbox name; replaces the legacy "edit `1. DEV` then stage" workflow.
9. **Metric parity / reconciliation**:
   - Dataworks test suite per migrated entity: row-count and `SUM(measure)` parity vs the live Synapse PRiME DW for the last full month.
   - Dual-load until two consecutive refresh cycles match within tolerance, then cut over the GMR dataset and decommission the corresponding `PRES_<DATASET>_DEV` views.
10. **Schema drift**: rely on Dataworks drift detection at the HLP layer; surface via CI test failures rather than silent updates to OCDM.

---

## 8. Dependencies

- **Data Integration**: `_ogr_azr_syd.<source>.*` populated, CDF on, `rtlh_row_hash` / `rtlh_ingestion_time` present, refresh SLA per source documented, communication of any hash-column redefinition (these appear as full reloads through CDF).
- **Dataworks framework**: `PersistedCDF`, `FullMerge`, `DeltaLoadUsingCT` notebooks; SCD2 / hash-SK helpers; Key Vault integration; DAB + Flyway packaging; published version on Artifactory pinned per OGR project.
- **OCDM (Erwin) model**: published, with SCD type per attribute and a stable change-management process; mapping artefact (`config/ocdm_mapping.yml`) generated from it.
- **Unity Catalog**: `prime_silver`, `prime_gold` catalogs with dataset-aligned schemas plus `sandbox_<dev>_*` for developer isolation; OGR group permissions.
- **GMR / Tabular**: re-pointing to Databricks SQL Warehouse.
- **CI/CD**: Azure DevOps pipelines re-templated for Dataworks (DAB + Flyway), replacing the DACPAC flow in PRiME.CICD.DW.yml and the workaround in PRIME.CICD.DW-uses local server.yml.

---

## 9. Reference example — GTUM (HME Group Time Usage Model)

Legacy chain (Synapse):

```
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DOWNTIMEAGGBYDAY
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DIMCAUSE
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DIMCAUSECATEGORY
   ↓ HLP_DEV.vw_hstg/...dim_FixedPlantTimeUsageModel + rep_/vw_cdc_ stack
HLP_DEV.dim_*, HLP_DEV.fact_HMETimeUsageEvent  (Synapse Data Flow sinks)
   ↓ PRES_DEV.fact_HMETimeUsageEvent  (OCDM, record grain)
PRES_HME_DEV.fact_HMEGroupTimeUsageEvent  (GROUP BY group-event keys)
   ↓ DS Tabular / GMR
```

Target chain — Dataworks project `ogr_hme`:

| # | Dataworks asset | Tooling | UC location | Replaces |
|---|---|---|---|---|
| 1 | *(out of scope, DI)* | — | `_ogr_azr_syd.udltimeutilisation.*` | UDL mocks + `rep_*` / `vw_cdc_*` |
| 2 | `notebooks/hlp/hlp_hme_timeusagemodel` | `PersistedCDF` + Flyway DDL | `prime_silver.hlp_hme.dim_timeusagemodel` | HLP_DEV.vw_HSTG_UDLTIMEUTILISATION_dim_FixedPlantTimeUsageModel.sql + `vw_dim_*` + `dim_*` SK table |
| 3 | `notebooks/hlp/hlp_hme_timeusageevent` | `PersistedCDF` + Flyway DDL | `prime_silver.hlp_hme.fact_timeusageevent` | `HLP_DEV.vw_fact_HMETimeUsageEvent_*` + `fact_HMETimeUsageEvent` |
| 4 | `notebooks/ocdm/ocdm_dim_hmetum` | `FullMerge` SCD2 + hash SK | `prime_silver.ocdm.dim_hmetum` | OCDM dim per Erwin |
| 5 | `notebooks/ocdm/ocdm_fact_hmetimeusageevent` | Incremental merge, month-partitioned | `prime_silver.ocdm.fact_hmetimeusageevent` | `PRES_DEV.fact_HMETimeUsageEvent` / `m_fact_*` |
| 6 | `flyway/R__view_gold_hme_fact_hmegrouptimeusageevent.sql` | Flyway repeatable view | `prime_gold.hme.fact_hmegrouptimeusageevent` | PRES_HME_DEV.fact_HMEGroupTimeUsageEvent.sql (same `GROUP BY`) |
| 7 | GMR `HME Group Event TUM` | Tabular re-point | semantic layer | Same metric, gold-pointed |

DAB job graph: `hlp_hme_timeusagemodel` ∥ `hlp_hme_timeusageevent` → `ocdm_dim_hmetum` → `ocdm_fact_hmetimeusageevent` → gold view refresh → reconciliation tests.

Reconciliation tests in the bundle:
- Row-count and `SUM(cGroupEventTUM)`, `SUM(sGroupEventTUM)` parity per `GroupEventStartDate` for last full month vs Synapse `PRES_HME_DEV.fact_HMEGroupTimeUsageEvent`.
- GMR metric parity in Power BI for the standard HME Time Usage report pages.
- Two consecutive matching refresh cycles → cut over → decommission Synapse `PRES_HME_DEV` HME views.

---

## 10. Acceptance-criteria coverage

| Criterion | Section |
|---|---|
| Transformation, not lift-and-shift | §3, §5 |
| Dataworks confirmed as standard framework (DAB + Flyway + reusable notebooks) | §3, §4.1, §5 |
| End-to-end flow incl. ingestion → OCDM → star → GMR | §3 diagram |
| Dataworks patterns for ingest consumption + OCDM conformance | §5.1–5.3 |
| Relationship Dataworks ↔ OCDM ↔ star ↔ GMR | §6 |
| Guidelines for SQL re-implementation + metric parity / reconciliation | §7 |
| Dependencies (Dataworks, OCDM/Erwin, DI source contract) | §8 |
| GTUM reference example | §9 |
| Documented for tech-lead sign-off | this doc |

# OGR – Dataworks Transformation Approach (Final, ODW v2 aligned)

Updates: Dataworks 2.0 introduces **ODW (OCDM on Databricks)** with hashed SKs, `sys_instance_id` salting, and OCDM naming/datatype conventions. The OGR target model is now an **ODW schema (`odw_<site>`)**, not a generic `prime_silver.ocdm` schema.

---

## 1. Current PRiME / Synapse DW (recap)

Layered Synapse SQL solution under 1. DEV:

- **UDL (mocked HSTG_V / DAL_V)** → **HLP_DEV** (per-source `vw_hstg` / `vw_dim` / `vw_fact` + custom CDC via `rep_*` + `vw_cdc_*` + `cdc.*`) → **PRES_DEV** (OCDM-aligned `dim_*`, `fact_*`, `m_fact_*`) → **PRES_<DATASET>_DEV** (HME / RAIL / EEDRILL / EEDRILLCOST / AM / PROCESS / PROCESSASSURANCE / GLOBAL_METRICS star schemas) → **DS Tabular models / GMR**.
- Refresh = Synapse Data Flows driven by Excel-backed metadata in MiDAS.
- Deploy = DACPAC compare → SQL script → manual SSMS run, with folder staging via Stage SQL Objects.ps1.

Pain to remove: Synapse Dedicated Pool cost, DACPAC workaround, hand-rolled CDC, IDENTITY-based SKs (RemoveIdentityColumn.sql), `1. DEV / 2. TEST / 3. PROD` folder staging, MiDAS dataflow metadata, no developer sandboxing, mixed-case PascalCase naming.

---

## 2. Source contract from Data Integration (out of OGR scope)

- Catalog **`_ogr_azr_syd`**, schema per source system; same `DAL_V.*` / `HSTG_V.*` entities used today.
- Each table is **Delta with CDF enabled** plus:
  - `rtlh_row_hash` — hash of business columns,
  - `rtlh_ingestion_time` — last ingest timestamp.
- Refresh MERGE keyed on `rtlh_row_hash`: unchanged → no-op; changed → old hash deleted + new hash inserted; dropped → deleted; new → inserted.

Implications:
1. Custom CDC (`rep_*`, `vw_cdc_*`, `cdc.*`) is replaced by `table_changes('_ogr_azr_syd.<src>.<tbl>', start_version)` consumed via Dataworks **`PersistedCDF`**.
2. Updates appear as `(delete, insert)` pairs at the same `_commit_version` — never `update_preimage` / `update_postimage`.
3. Join downstream on **business keys**, not `rtlh_row_hash` (the hash changes when business data changes).
4. Hard deletes propagate → ODW dim SCD logic must close current rows / set `is_deleted` (replaces `OMD_DELETED_RECORD_INDICATOR`).
5. UDL mocking workflow (`[HLP_DEV].[sp_UDLVWDesc]`) is obsolete.

---

## 3. Target architecture on Databricks + Dataworks 2.0 (ODW)

```mermaid
flowchart LR
    subgraph DI["Data Integration (out of OGR scope)"]
        SRC[Source systems]
        ING["_ogr_azr_syd.&lt;source&gt;.*<br/>Delta + CDF<br/>rtlh_row_hash / rtlh_ingestion_time"]
        SRC --> ING
    end

    subgraph OGR["OGR Dataworks projects (one per dataset)"]
      direction TB
      subgraph HLP["prime_silver.hlp_&lt;source&gt;"]
        H1[Per-source shaping<br/>PersistedCDF / FullMerge]
      end
      subgraph ODW["odw_&lt;site&gt; (OCDM on Databricks)"]
        O1["dim_* (snake_case)<br/>SK = xxhash64(sys_instance_id | bk_cols)"]
        O2["fact_* / m_fact_*<br/>partition by event-month"]
      end
      subgraph GOLD["prime_gold.&lt;dataset&gt; (Star schema for consumers)"]
        G1[dim_* / fact_* per OGR dataset]
      end
    end

    subgraph Sem["Semantic layer"]
        M1[GMR metrics / Tabular / Power BI on Databricks SQL Warehouse]
    end

    ING --> HLP --> ODW --> GOLD --> Sem

    Erwin[(OCDM<br/>Erwin)] -.contract.-> ODW
    InstanceReg[(ODP Instance ID registry)] -.sys_instance_id.-> ODW
    Flyway[Flyway CLI<br/>DDL migrations] -.deploys.-> HLP
    Flyway -.deploys.-> ODW
    Flyway -.deploys.-> GOLD
    DAB[Databricks Asset Bundle<br/>jobs / notebooks] -.orchestrates.-> HLP
    DAB -.orchestrates.-> ODW
    DAB -.orchestrates.-> GOLD
    KV[Azure Key Vault] -.auth.-> DAB
```

| Target | Replaces | Owner / Tooling |
|---|---|---|
| `_ogr_azr_syd.<source>.*` | `HSTG_V.*` / `DAL_V.*` + UDL mocks | Data Integration |
| `prime_silver.hlp_<source>.*` | `HLP_DEV.vw_hstg/vw_dim/vw_fact` + `rep_*`/`vw_cdc_*`/`cdc.*` | OGR Dataworks (`PersistedCDF` + Flyway) |
| **`odw_<site>.dim_*` / `fact_*`** | `PRES_DEV.dim_*/fact_*/m_fact_*` | OGR Dataworks (Erwin OCDM contract, ODW v2 conventions) |
| `prime_gold.<dataset>.*` | `PRES_<DATASET>_DEV.*` | OGR Dataworks |
| GMR / Tabular | DS Tabular models in DS/ | Re-pointed to Databricks SQL Warehouse |

Site code TBD with platform (e.g. `odw_wei` per the Dataworks doc); a unique `sys_instance_id` must be allocated from the ODP Instance ID registry before any ODW DDL is deployed.

---

## 4. ODW conventions adopted (mandatory)

Applied uniformly across the `odw_<site>` and dependent gold layer:

- **Schema naming**: `odw_<site_code>` (e.g. `odw_wei`).
- **Table naming**: snake_case (`dim_site`, `fact_fixed_plant_time_usage_event`). The legacy PascalCase names (`fact_HMEGroupTimeUsageEvent`) are **renamed** at the ODW boundary; the gold layer can re-expose backwards-compatible aliases as Flyway `R__` views during cut-over to avoid GMR breakage.
- **Column naming**: snake_case (`site_sk`, `fixed_plant_time_usage_event_sk`, `time_usage_explanation`).
- **Surrogate keys** (Fact & Dim):
  - SQL: `xxhash64(concat_ws('|', sys_instance_id, <bk_col_1>, <bk_col_2>, ...)) AS <entity>_sk`
  - Python: `xxhash.xxh64_intdigest('|'.join([sys_instance_id, *bk_cols]))`
  - Removes IDENTITY plumbing entirely (no RemoveIdentityColumn.sql equivalent needed) and supports **fact-first ETL with stub rows** for late-arriving dims.
- **`sys_instance_id`** present on every ODW table; included in every SK hash.
- **Datetime handling**:
  - System columns: `timestamp` (UTC) — `sys_created_datetime_utc`, `sys_updated_datetime_utc`.
  - Business columns: `timestamp_ntz` in local time — `record_event_datetime`, `base_event_datetime`, `group_event_datetime`.
  - Date SK columns: `date` in local time — `record_event_start_date_sk`, etc.
  - Every ODW table carries `sys_timezone_name` (IANA, e.g. `Australia/Perth`).
- **Soft-delete / SCD**: `is_deleted` + SCD2 close-row pattern in `dim_*`; default Type-1 unless Erwin marks the attribute historised.

The legacy Synapse `OMD_*` columns and `*_SK` suffixes survive only inside the HLP layer for traceability; they are **not** carried into `odw_<site>`.

---

## 5. Standard Dataworks project layout (per OGR dataset)

```
ogr_hme/
├── databricks.yml                  # DAB root (targets: dev, test, prod, sandbox-<dev>)
├── resources/                      # DAB jobs, schedules, clusters/SQL warehouses
│   └── ogr_hme.job.yml
├── notebooks/
│   ├── hlp/                        # per-source shaping (PersistedCDF)
│   ├── odw/                        # ODW dims/facts (xxhash64 SKs, sys_instance_id)
│   └── gold/                       # star-schema projections / role-playing splits
├── flyway/
│   ├── V001__create_schemas.sql           # odw_<site>, prime_silver.hlp_<src>, prime_gold.<ds>
│   ├── V010__odw_<site>_dims.sql          # dim_* DDL with sys_instance_id, sys_timezone_name
│   ├── V020__odw_<site>_facts.sql         # fact_* DDL, partition by event-month
│   └── R__gold_<ds>_views.sql             # repeatable views (incl. legacy-name aliases)
├── config/
│   ├── ingestion.yml               # HLP source list + bk columns
│   └── ocdm_mapping.yml            # OCDM/Erwin-derived mapping (bk cols → SK)
├── tests/                          # row-count, sum, SK uniqueness, reconciliation
└── pyproject.toml                  # pins Dataworks 2.x package from Artifactory
```

Environments are **DAB targets**, not folders → eliminates the `1. DEV / 2. TEST / 3. PROD` staging.

---

## 6. Patterns per layer

### 6.1 HLP — per-source shaping

- **`PersistedCDF`** notebook parameterised by `(source_table, target_table, business_key, transform_sql)`.
- Bookmark = last `_commit_version` per (source, target).
- `(delete, insert)` pairs at same commit version → logical update; handled by `MERGE` on the deterministic hash SK.
- One HLP model per current `HLP_DEV` view → 1:1 reconciliation.
- Schema drift handled by Dataworks framework.
- HLP retains source-style column names; conversion to ODW snake_case happens in the ODW layer.

### 6.2 ODW — `odw_<site>` (OCDM on Databricks)

- One Dataworks notebook per Erwin entity → `odw_<site>.dim_*` / `fact_*`.
- **Surrogate key** generated as ODW v2 prescribes:
  ```sql
  xxhash64(concat_ws('|', sys_instance_id, <bk_col_1>, <bk_col_2>, ...)) as <entity>_sk
  ```
- BK columns are explicit `<col>_bk` columns sourced from HLP.
- SCD2 via Dataworks `FullMerge` macro pattern, on Erwin-flagged historised attributes only.
- Hard deletes → `is_deleted = true` + close current SCD2 row.
- Facts partitioned by event-month (local time `*_date_sk`); `m_fact` materialisation = partition filter `>= add_months(current_date(), -6)` + `OPTIMIZE … ZORDER BY` on hot dim FKs. Replaces Load fact partition.sql.
- Datetime columns follow ODW datatype rules (`timestamp` UTC for `sys_*`, `timestamp_ntz` local for business, `date` local for `*_date_sk`); `sys_timezone_name` populated from site config.

### 6.3 Gold — per-dataset star schema

- One Dataworks notebook (or Flyway `R__` view) per current `PRES_<DATASET>_DEV` object.
- Purpose: thin projections, role-playing splits (e.g. `fact_HMEBaseTimeUsageEvent`, `fact_HMEGroupTimeUsageEvent`, `fact_HMERecordTimeUsageEvent` from one ODW fact), legacy-name aliases for GMR continuity during cut-over.
- Strip technical columns (`sys_*` except where needed by GMR).

### 6.4 Orchestration, environments, security

- DAB job per dataset wires HLP → ODW → Gold tasks.
- Targets: `dev`, `test`, `prod`, `sandbox-<dev>`. Sandboxes use a developer-prefixed catalog/schema.
- Auth: Azure Key Vault via Dataworks integration.
- CI/CD pipelines in CICD/ re-templated to run `dataworks build` → `flyway migrate` → `databricks bundle deploy --target <env>`. Replaces the DACPAC + GO-stripping workaround in PRIME.CICD.DW-uses local server.yml.

---

## 7. Relationship between artefacts

| Artefact | Owns | Drives |
|---|---|---|
| **Dataworks project (DAB + Flyway + reusable notebooks)** | All OGR code, DDL, and orchestration. | Build order, tests, multi-env deploy, sandboxes. |
| **OCDM model in Erwin** | Logical contract for `odw_<site>.*`. | Entity / attribute names, datatypes, BK columns, SCD type, conformed dims. |
| **ODP Instance ID registry** | `sys_instance_id` per ODW instance. | SK hashing salt; cross-site uniqueness. |
| **Star schema (`prime_gold.<ds>`)** | Physical analytical model per OGR dataset. | Query shape, role-playing dims, legacy-name aliases. |
| **GMR metrics** | Business KPI definitions in semantic layer. | Tabular measures + parity tests. |

---

## 8. Reimplementation guidelines

1. **Out of scope for OGR**: UDL mocked views, `[HLP_DEV].[sp_UDLVWDesc]`, all `rep_*` / `vw_cdc_*` / `cdc.*` objects, `1. DEV / 2. TEST / 3. PROD` staging.
2. **Per legacy SQL file**: classify into HLP / ODW / gold and create the matching Dataworks asset (notebook + Flyway DDL).
3. **Naming**: all ODW objects in snake_case per ODW conventions. Legacy PascalCase names survive only as gold-layer aliases during cut-over.
4. **Surrogate keys**: ODW v2 hash pattern with `sys_instance_id` salt and pipe-separated BK columns. No IDENTITY columns anywhere.
5. **CDF consumption**: `PersistedCDF`; persist `_commit_version` bookmark; treat `(delete, insert)` pairs as logical update; never join on `rtlh_row_hash`.
6. **SCD**: default Type-1; Type-2 only where Erwin marks the attribute historised; deletes → `is_deleted = true` + close current row.
7. **Datetime**: enforce ODW datatype split (`timestamp` UTC for system, `timestamp_ntz` local for business, `date` local for `*_date_sk`); always populate `sys_timezone_name`.
8. **Sandbox first**: developers work in `sandbox-<name>` DAB targets writing to dev catalogs; replaces the legacy "edit `1. DEV` then stage" workflow.
9. **Metric parity / reconciliation**:
   - Dataworks test suite per migrated entity: row-count and `SUM(measure)` parity vs the live Synapse PRiME DW for the last full month.
   - Dual-load until two consecutive refresh cycles match within tolerance, then cut over the GMR dataset and decommission the corresponding `PRES_<DATASET>_DEV` views.
10. **Schema drift**: rely on Dataworks drift detection at HLP; surface via CI test failures rather than silent ODW updates.

---

## 9. Dependencies

- **Data Integration**: `_ogr_azr_syd.<source>.*` populated, CDF on, `rtlh_row_hash` / `rtlh_ingestion_time` present, refresh SLA per source documented; communication of any hash-column redefinition (these appear as full reloads through CDF).
- **Dataworks 2.0+**: `PersistedCDF`, `FullMerge`, `DeltaLoadUsingCT`, ODW SK helpers, schema-drift detection, Key Vault integration, DAB + Flyway packaging; version pinned per OGR project from Artifactory.
- **OCDM model in Erwin**: published, with BK columns, SCD type per attribute, and a stable change-management process; mapping artefact (`config/ocdm_mapping.yml`) generated from it.
- **ODP Instance ID** allocated for the OGR ODW instance (registered in the ODP Instance IDs list).
- **Unity Catalog**: `prime_silver`, `odw_<site>`, `prime_gold` catalogs/schemas with OGR group permissions and `sandbox_<dev>_*` schemas for developer isolation.
- **GMR / Tabular**: re-pointing to Databricks SQL Warehouse.
- **CI/CD**: Azure DevOps pipelines re-templated for Dataworks (DAB + Flyway), replacing the DACPAC flow in PRiME.CICD.DW.yml.

---

## 10. Reference example — GTUM (HME Group Time Usage Model)

Legacy chain (Synapse):

```
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DOWNTIMEAGGBYDAY
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DIMCAUSE
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DIMCAUSECATEGORY
   ↓ HLP_DEV.vw_hstg/...dim_FixedPlantTimeUsageModel + rep_/vw_cdc_ stack
HLP_DEV.dim_*, HLP_DEV.fact_HMETimeUsageEvent  (Synapse Data Flow sinks)
   ↓ PRES_DEV.fact_HMETimeUsageEvent  (OCDM, record grain)
PRES_HME_DEV.fact_HMEGroupTimeUsageEvent  (GROUP BY group-event keys)
   ↓ DS Tabular / GMR
```

Target chain — Dataworks project `ogr_hme`, ODW instance `odw_<site>`:

| # | Dataworks asset | Tooling | UC location | Replaces |
|---|---|---|---|---|
| 1 | *(out of scope, DI)* | — | `_ogr_azr_syd.udltimeutilisation.*` | UDL mocks + `rep_*` / `vw_cdc_*` |
| 2 | `notebooks/hlp/hlp_hme_time_usage_model` | `PersistedCDF` + Flyway DDL | `prime_silver.hlp_hme.dim_time_usage_model` | HLP_DEV.vw_HSTG_UDLTIMEUTILISATION_dim_FixedPlantTimeUsageModel.sql + `vw_dim_*` + `dim_*` SK table |
| 3 | `notebooks/hlp/hlp_hme_time_usage_event` | `PersistedCDF` + Flyway DDL | `prime_silver.hlp_hme.fact_time_usage_event` | `HLP_DEV.vw_fact_HMETimeUsageEvent_*` + `fact_HMETimeUsageEvent` |
| 4 | `notebooks/odw/dim_hme_time_usage_model` | `FullMerge` SCD2 | `odw_<site>.dim_hme_time_usage_model` <br/>`hme_time_usage_model_sk = xxhash64(concat_ws('\|', sys_instance_id, tum5_code_bk, tum6_local_code_bk, tum7_local_code_bk))` | OCDM dim per Erwin |
| 5 | `notebooks/odw/fact_hme_time_usage_event` | Incremental merge, partition by `record_event_start_date_sk` month | `odw_<site>.fact_hme_time_usage_event` | `PRES_DEV.fact_HMETimeUsageEvent` / `m_fact_*` |
| 6 | `flyway/R__gold_hme_fact_hme_group_time_usage_event.sql` | Flyway repeatable view | `prime_gold.hme.fact_hme_group_time_usage_event` (alias view `fact_HMEGroupTimeUsageEvent` for GMR back-compat) | PRES_HME_DEV.fact_HMEGroupTimeUsageEvent.sql (same `GROUP BY`) |
| 7 | GMR `HME Group Event TUM` | Tabular re-point | semantic layer | Same metric, gold-pointed |

Sketch of the ODW SK in step 4:
```sql
SELECT
   xxhash64(concat_ws('|', sys_instance_id,
                      tum5_code_bk, tum6_local_code_bk, tum7_local_code_bk))
       AS hme_time_usage_model_sk,
   tum5_code, tum6_local_code, tum6_local_name,
   tum7_local_code, tum7_local_name, tum7_local_description,
   sys_instance_id,
   sys_created_datetime_utc, sys_updated_datetime_utc,
   sys_timezone_name,
   is_deleted
FROM prime_silver.hlp_hme.dim_time_usage_model;
```

DAB job graph: `hlp_hme_time_usage_model` ∥ `hlp_hme_time_usage_event` → `dim_hme_time_usage_model` → `fact_hme_time_usage_event` → gold view refresh → reconciliation tests.

Reconciliation tests in the bundle:
- Row-count and `SUM(c_group_event_tum)`, `SUM(s_group_event_tum)` parity per `group_event_start_date_sk` for last full month vs Synapse `PRES_HME_DEV.fact_HMEGroupTimeUsageEvent`.
- SK uniqueness on every ODW dim/fact.
- GMR metric parity in Power BI for the standard HME Time Usage report pages.
- Two consecutive matching refresh cycles → cut over → decommission Synapse `PRES_HME_DEV` HME views.

---

## 11. Acceptance-criteria coverage

| Criterion | Section |
|---|---|
| Transformation, not lift-and-shift | §3, §6 |
| Dataworks 2.0 (DAB + Flyway + reusable notebooks + ODW) confirmed as standard | §3, §4, §5, §6 |
| End-to-end flow incl. ingestion → ODW (OCDM) → star → GMR | §3 diagram |
| Dataworks patterns for ingest consumption + OCDM conformance | §6.1–6.3 |
| Relationship Dataworks ↔ OCDM (Erwin) ↔ ODW ↔ star ↔ GMR | §7 |
| Guidelines for SQL re-implementation + metric parity / reconciliation | §8 |
| Dependencies (Dataworks 2.0, OCDM/Erwin, Instance ID, DI source contract) | §9 |
| GTUM reference example (with ODW SK + naming) | §10 |
| Documented for tech-lead sign-off | this doc |

# OGR – Dataworks Transformation Approach (Final, ODW v2 + CDC + MiDAS + GMR + DW project structure)

Two corrections applied throughout:

1. **GMR = Global Metric Registry** — a Power Apps app where Power BI semantic-layer measures are defined in a two-tier model (**base measures** → **metrics**). It is *not* the Tabular model itself; the Tabular model is generated/aligned from GMR definitions. Renaming a gold table or column has a *registered* impact in GMR.
2. **Project layout aligned to the official Dataworks "Project Structure"** — `.rtlh_dataworks/`, `config/{json,sql}/`, `core/`, `devops_pipelines/`, `notebooks/`, `resources/{jobs,pipelines}/`, `sql_deployment/{Repeatable,V001}/`, `project.schema.toml`, `databricks.yml` — with Flyway versioned scripts using a unique project prefix.

---

## 1. Current PRiME / Synapse DW (recap)

Layered Synapse SQL solution under 1. DEV:

- **UDL (mocked HSTG_V / DAL_V)** → **HLP_DEV** (per-source `vw_hstg` / `vw_dim` / `vw_fact` + custom CDC via `rep_*` + `vw_cdc_*` + `cdc.*` + MiDAS-driven `mde` views) → **PRES_DEV** (OCDM-aligned `dim_*`, `fact_*`, `m_fact_*`) → **PRES_<DATASET>_DEV** (HME / RAIL / EEDRILL / EEDRILLCOST / AM / PROCESS / PROCESSASSURANCE / GLOBAL_METRICS star schemas) → **DS Tabular models / GMR**.
- **MiDAS** = Power Apps app — owns dataflow / dataflow-dependency metadata that orchestrates Synapse Data Flows, **and** is the manual-data-entry front-end (consumed by `HLP_DEV.mde.*`).
- **GMR (Global Metric Registry)** = Power Apps app — owns Power BI semantic-layer definitions in two tiers: **base measures** (1:1 with gold columns/aggregates) and **metrics** (composed from base measures and other metrics). Tabular models in DS/ are aligned to GMR.
- Refresh = Synapse Data Flows orchestrated by metadata in MiDAS.
- Deploy = DACPAC compare → SQL script → manual SSMS run, with folder staging via Stage SQL Objects.ps1.

### 1.1 Legacy CDC / refresh tiers

| # | Tier | Where | What it does today | Limits |
|---|---|---|---|---|
| 1 | **UDL Extractor** | Source → UDL `DAL` / `HSTG` | Full load most tables; rolling 3–6 month windows for very large tables (Cycle / Event from ORDW & MMRS). | Window-based, not true CDC. |
| 2 | **HLP CDC** | UDL → `HLP_DEV.dim_*` and CRI `HLP_DEV.fact_*` | Full compare on dims (so SKs persist for facts); partial 3-month compare on CRI fact tables to capture Work-Day-5 snapshot timestamp; auto-falls-back to full when target empty. | Bespoke `rep_*` + `vw_cdc_*` machinery. |
| 3 | **PRES Partition Refresh** | HLP → `PRES_DEV.fact_*` | Used when HLP CDC is *not* used. Source-system or source-system+year partitioning; daily refresh of last 2 quarters for big facts. | Manual partition load procs ([Load fact partition.sql](DW/Scripts/Load%20fact%20partition.sql)). |
| 4 | **Dataset Incremental Refresh** | PRES → Tabular / Power BI | HME Cycle/Event refresh last 2 quarters daily; everything else fully refreshed. | Refresh-time bottleneck. |

All four tiers are driven by metadata in MiDAS.

Pain to remove: bespoke HLP CDC, two parallel "CDC vs partition refresh" patterns, IDENTITY-based SKs (RemoveIdentityColumn.sql), `1. DEV / 2. TEST / 3. PROD` folder staging, **MiDAS as orchestration source of truth**, manual partition loaders, dataset-level incremental refresh as the only way to keep BI fast.

---

## 2. Source contract from Data Integration (out of OGR scope)

- Catalog **`_ogr_azr_syd`**, schema per source system.
- Each table is **Delta with CDF enabled** plus:
  - `rtlh_row_hash` — hash of business columns,
  - `rtlh_ingestion_time` — last ingest timestamp.
- Refresh MERGE keyed on `rtlh_row_hash`: unchanged → no-op; changed → old hash deleted + new hash inserted; dropped → deleted; new → inserted.

This single tier replaces the **UDL Extractor + HLP CDC** combo:

1. Custom CDC (`rep_*`, `vw_cdc_*`, `cdc.*`) is replaced by `table_changes('_ogr_azr_syd.<src>.<tbl>', start_version)` consumed via Dataworks **`PersistedCDF`**.
2. Updates appear as `(delete, insert)` pairs at the same `_commit_version` — never `update_preimage` / `update_postimage`.
3. Join downstream on **business keys**, not `rtlh_row_hash`.
4. Hard deletes propagate → ODW dim SCD logic must close current rows / set `is_deleted` (replaces `OMD_DELETED_RECORD_INDICATOR`).
5. Window-based extracts (last 3/6 months for Cycle/Event) are no longer needed.
6. UDL mocking workflow (`[HLP_DEV].[sp_UDLVWDesc]`) is obsolete.

### 2.1 MiDAS in the new world

- **Manual data entry** — unchanged. MDE tables ingested into `_ogr_azr_syd.midas.*` by Data Integration on the same CDF contract; OGR consumes via `prime_silver.hlp_midas.*` (replaces `HLP_DEV.mde.*`).
- **Orchestration metadata** — **no longer used by OGR**. Job graph, schedules, dependencies, environments live in the Dataworks DAB (`databricks.yml` + `resources/jobs/*.yml`) in Git.

### 2.2 GMR in the new world

- **Unchanged role**: continues to own Power BI base measures and metrics (the two-tier semantic definition).
- **What changes**: the gold-layer columns GMR base measures point at move from Synapse (`PRES_<DATASET>_DEV.*`) to Databricks (`prime_gold.<dataset>.*`). To minimise GMR re-registration:
  - Keep gold column names stable.
  - Where ODW snake_case differs from legacy PascalCase, expose **legacy-name alias columns** in the gold view layer during cut-over.
  - Treat GMR base-measure definitions as the contract that drives the gold layer's column surface — any rename requires a coordinated GMR change.

---

## 3. Target architecture on Databricks + Dataworks 2.0 (ODW)

```mermaid
flowchart LR
    subgraph DI["Data Integration (out of OGR scope)"]
        SRC[Source systems]
        MIDAS["MiDAS (Power Apps)<br/>Manual Data Entry"]
        ING["_ogr_azr_syd.&lt;source&gt;.*<br/>incl. _ogr_azr_syd.midas.*<br/>Delta + CDF"]
        SRC --> ING
        MIDAS --> ING
    end

    subgraph OGR["OGR Dataworks projects (one per data product)"]
      direction TB
      subgraph HLP["prime_silver.hlp_&lt;source&gt;"]
        H1[Per-source shaping<br/>PersistedCDF / FullMerge]
      end
      subgraph ODW["odw_&lt;site&gt; (OCDM on Databricks)"]
        O1["dim_* (snake_case)<br/>SK = xxhash64(sys_instance_id | bk_cols)"]
        O2["fact_* / m_fact_*<br/>partition by event_month"]
      end
      subgraph GOLD["prime_gold.&lt;dataset&gt; (Star schema)"]
        G1[dim_* / fact_* per OGR dataset<br/>+ legacy-name alias columns]
      end
    end

    subgraph Sem["Semantic layer"]
        GMR["GMR (Power Apps)<br/>base measures → metrics"]
        TM[Tabular model / Power BI<br/>on Databricks SQL Warehouse]
        GMR --> TM
    end

    ING --> HLP --> ODW --> GOLD --> TM

    Erwin[(OCDM<br/>Erwin)] -.contract.-> ODW
    InstanceReg[(ODP Instance ID registry)] -.sys_instance_id.-> ODW
    Flyway[Flyway CLI<br/>sql_deployment/] -.deploys.-> HLP
    Flyway -.deploys.-> ODW
    Flyway -.deploys.-> GOLD
    DAB[DAB CLI<br/>resources/jobs/] -.orchestrates.-> HLP
    DAB -.orchestrates.-> ODW
    DAB -.orchestrates.-> GOLD
    KV[Azure Key Vault] -.auth.-> DAB
```

### 3.1 Tier-by-tier mapping (legacy → target)

| Legacy tier | Target on Databricks | Notes |
|---|---|---|
| UDL Extractor | DI's CDF-enabled `_ogr_azr_syd.<src>.*` | Single ingestion contract. |
| HLP CDC – dimensions | `prime_silver.hlp_<src>` (`PersistedCDF`) → `odw_<site>.dim_*` with deterministic hash SK | SK stable by construction. |
| HLP CDC – CRI fact tables (Work-Day-5 snapshots) | `prime_silver.hlp_cri.fact_*` consuming CDF; `sys_first_seen_datetime_utc` column captures snapshot timestamp. | Partial vs full = bookmark concern. |
| PRES Partition Refresh | `odw_<site>.fact_*` partitioned by `event_month` (+ `sys_instance_id` for big facts); incremental MERGE; `m_fact_*` = last-6-months hot. | One pattern for all facts. |
| Dataset Incremental Refresh | Direct Lake / DirectQuery against gold; or Import + IR aligned to `event_month`. | Incremental moves to the warehouse. |
| MiDAS dataflow / dependency metadata | DAB `databricks.yml` + `resources/jobs/*.yml` in Git | Removes dual source of truth. |
| MiDAS manual data entry | **Unchanged** — MDE retained; ingested as `_ogr_azr_syd.midas.*`. | Read via `prime_silver.hlp_midas.*`. |
| GMR base measures / metrics | **Unchanged role** — re-pointed to `prime_gold.<dataset>.*` columns; legacy column-name aliases preserve registrations during cut-over. | Renames require coordinated GMR change. |

---

## 4. ODW conventions adopted (mandatory)

- **Schema**: `odw_<site_code>` (e.g. `odw_wei`).
- **Tables / columns**: snake_case (`fact_fixed_plant_time_usage_event`, `site_sk`).
- **Surrogate keys** (Fact & Dim):
  - SQL: `xxhash64(concat_ws('|', sys_instance_id, <bk_col_1>, <bk_col_2>, ...)) AS <entity>_sk`
  - Removes IDENTITY plumbing; supports **fact-first ETL with stub rows** for late-arriving dims.
- **`sys_instance_id`** on every ODW table; included in every SK hash.
- **Datetime**:
  - System columns: `timestamp` UTC — `sys_created_datetime_utc`, `sys_updated_datetime_utc`.
  - Business columns: `timestamp_ntz` local — `record_event_datetime`, etc.
  - Date SK columns: `date` local — `record_event_start_date_sk`, etc.
  - `sys_timezone_name` (IANA) on every ODW table.
- **Soft-delete / SCD**: `is_deleted` + SCD2 close-row pattern; default Type-1 unless Erwin marks the attribute historised.
- **CRI snapshot semantics**: `sys_first_seen_datetime_utc` column on relevant ODW facts.

Legacy `OMD_*` / `CDC_*` columns are kept only in HLP for traceability, not propagated to ODW.

---

## 5. Standard Dataworks project layout (per OGR data product)

Aligned with the official Dataworks "Project Structure" guide. Example: `data_products/ogr_hme/`.

```
data_products/ogr_hme/
├── .rtlh_dataworks/                 # reserved – reusable Dataworks components
├── config/
│   ├── json/                        # config (e.g. ingestion bookmarks, run params)
│   │   └── ingestion.json
│   └── sql/                         # source-extract & transform SQL referenced by notebooks
│       ├── hlp_hme_time_usage_model.sql
│       └── hlp_hme_time_usage_event.sql
├── core/                            # OGR-specific reusable Python / notebooks (e.g. SK helpers,
│                                    # CRI sys_first_seen_datetime_utc helper)
├── devops_pipelines/
│   └── ogr_hme.azure-pipelines.yml  # CI/CD (replaces CICD/PRiME.CICD.DW.yml)
├── notebooks/
│   ├── hlp/                         # PersistedCDF-driven shaping; incl. hlp_midas for MDE
│   ├── odw/                         # ODW dims/facts (xxhash64 SKs, sys_instance_id)
│   └── gold/                        # star-schema projections / role-playing splits / GMR aliases
├── resources/
│   ├── jobs/
│   │   └── ogr_hme.job.yml          # DAB workflow (replaces MiDAS dataflow metadata)
│   └── pipelines/                   # DLT pipeline configs (only if used)
├── sql_deployment/                  # Flyway DDL
│   ├── Repeatable/
│   │   └── R__gold_hme_views.sql    # incl. legacy-name alias views/columns for GMR
│   └── V001/
│       ├── V0001_3000000001__ogr_hme_create_schemas.sql
│       ├── V0001_3000000002__ogr_hme_odw_dims.sql
│       └── V0001_3000000003__ogr_hme_odw_facts.sql
├── project.schema.toml              # schema definitions for the data product
└── databricks.yml                   # DAB root (targets: dev, test, prod, sandbox-<dev>)
```

Notes:

- Versioned Flyway scripts use a **unique project prefix** in the version number to avoid collisions across OGR data products. Allocate a project-prefix range per OGR product (e.g. `ogr_hme = 3xxxxxxxxx`, `ogr_rail = 4xxxxxxxxx`, …) following the Dataworks recommendation `V0001_<project_prefix>...__<description>.sql`.
- One project per OGR **data product** (HME, RAIL, EEDRILL, EEDRILLCOST, AM, PROCESS, PROCESSASSURANCE, GLOBAL_METRICS), each scaffolded via `dataworks create` so the structure stays consistent.
- Environments are **DAB targets in `databricks.yml`**, not folders → eliminates `1. DEV / 2. TEST / 3. PROD` staging.

---

## 6. Patterns per layer

### 6.1 HLP — per-source shaping (replaces UDL Extractor + HLP CDC + `mde` views)

- `notebooks/hlp/*` use `PersistedCDF` from `.rtlh_dataworks/`, parameterised by `(source_table, target_table, business_key, transform_sql, [event_date_filter])`. The `transform_sql` lives under `config/sql/`.
- Bookmark = last `_commit_version` per (source, target).
- `(delete, insert)` pairs at same commit version → logical update; handled by `MERGE` on the deterministic hash SK.
- One HLP model per current `HLP_DEV` view → 1:1 reconciliation.
- **MDE handling**: `prime_silver.hlp_midas.*` models consume `_ogr_azr_syd.midas.*` and replicate today's `HLP_DEV.mde.*`.
- For CRI: HLP captures `sys_first_seen_datetime_utc = least(existing, current_ingest_time)` for Work-Day-5 snapshot semantics.

### 6.2 ODW — `odw_<site>` (replaces HLP CDC dim persistence + PRES OCDM views)

- One `notebooks/odw/*` per Erwin entity → `odw_<site>.dim_*` / `fact_*`.
- SK = `xxhash64(concat_ws('|', sys_instance_id, <bk_cols>))` — stable across full reloads, satisfying the legacy "must persist dim SK" need without bespoke CDC.
- SCD2 via Dataworks `FullMerge` macro pattern, on Erwin-flagged historised attributes only; default Type-1.
- Hard deletes → `is_deleted = true` + close current SCD2 row.
- Facts partitioned by event-month; big facts (HME Cycle/Event, MMRS Event) additionally partitioned by `sys_instance_id`.
- `m_fact_*` = last-6-months partition filter + `OPTIMIZE … ZORDER BY` on hot dim FKs. Replaces Load fact partition.sql.
- DDL maintained under `sql_deployment/V001/` (Flyway).

### 6.3 Gold — per-dataset star schema (replaces `PRES_<DATASET>_DEV`)

- One `notebooks/gold/*` (or `sql_deployment/Repeatable/R__*` view) per current `PRES_<DATASET>_DEV` object.
- Thin projections, role-playing splits.
- **GMR contract**: gold column names == GMR base-measure registrations. Legacy PascalCase exposed as alias columns/views during cut-over so existing GMR base measures continue to bind without re-registration.

### 6.4 Replacing dataset Incremental Refresh

- Default = **DirectQuery / Direct Lake** from Tabular against gold.
- If Import remains, configure Power BI Incremental Refresh on the gold table's `event_month` column.

### 6.5 Orchestration, environments, security

- **DAB job per data product** (`resources/jobs/<product>.job.yml`) wires HLP → ODW → Gold tasks **and owns dependencies + schedules** (replacing MiDAS dataflow metadata).
- Targets in `databricks.yml`: `dev`, `test`, `prod`, `sandbox-<dev>`.
- Auth: Azure Key Vault via Dataworks integration.
- CI/CD pipelines under `devops_pipelines/` run `dataworks build` → `flyway migrate` → `databricks bundle deploy --target <env>`. Replaces the DACPAC + GO-stripping workaround in PRIME.CICD.DW-uses local server.yml.

---

## 7. Relationship between artefacts

| Artefact | Owns | Drives |
|---|---|---|
| **Dataworks project (DAB + Flyway + reusable notebooks)** | All OGR code, DDL, **and orchestration / dependencies** (the role MiDAS dataflow metadata played). | Build order, tests, multi-env deploy, sandboxes. |
| **OCDM model in Erwin** | Logical contract for `odw_<site>.*`. | Entity / attribute names, datatypes, BK columns, SCD type, conformed dims. |
| **ODP Instance ID registry** | `sys_instance_id` per ODW instance. | SK hashing salt; cross-site uniqueness. |
| **MiDAS (Power Apps)** | Manual data entry only in the new world. | Source of `_ogr_azr_syd.midas.*` → `prime_silver.hlp_midas.*`. |
| **Star schema (`prime_gold.<ds>`)** | Physical analytical model per OGR dataset. | Query shape, role-playing dims, **GMR-aligned column surface (incl. legacy aliases)**. |
| **GMR (Power Apps)** | Base measures (1:1 with gold columns) and metrics (composed). | Tabular / Power BI semantic layer; rename impact analysis. |

---

## 8. Reimplementation guidelines

1. **Out of scope for OGR**: UDL mocked views, `[HLP_DEV].[sp_UDLVWDesc]`, all `rep_*` / `vw_cdc_*` / `cdc.*` objects, `1. DEV / 2. TEST / 3. PROD` staging, the UDL Extractor, manual partition load procs, **MiDAS as orchestrator**.
2. **MiDAS treatment**: keep MDE; drop the orchestration metadata role. Map every `HLP_DEV.mde.*` view to a `prime_silver.hlp_midas.*` Dataworks model.
3. **GMR treatment**: keep base measures and metrics. Pre-migration, run a GMR impact analysis per gold table; expose legacy-name alias columns/views for any column that would otherwise be renamed by ODW snake_case conventions. Coordinate any genuine rename with the GMR owner.
4. **Per legacy SQL file**: classify into HLP / ODW / gold and create the matching Dataworks asset (notebook in `notebooks/`, DDL in `sql_deployment/`).
5. **Naming**: ODW objects in snake_case. Legacy PascalCase survives only as gold-layer aliases during cut-over.
6. **Surrogate keys**: ODW v2 hash with `sys_instance_id` salt. No IDENTITY columns. Stable across full reloads → safe for fact joins without bespoke CDC.
7. **CDF consumption**: `PersistedCDF`; persist `_commit_version` bookmark; treat `(delete, insert)` pairs as logical update; never join on `rtlh_row_hash`.
8. **CRI snapshot semantics**: capture `sys_first_seen_datetime_utc` in HLP.
9. **Big facts**: partition by `event_month` (and `sys_instance_id` if needed); incremental MERGE via CDF bookmark replaces "last 2 quarters daily" partition refresh.
10. **SCD**: default Type-1; Type-2 only where Erwin marks the attribute historised; deletes → `is_deleted = true` + close current row.
11. **Datetime**: enforce ODW datatype split; always populate `sys_timezone_name`.
12. **Flyway versioning**: use the project-prefix convention (`V0001_<project_prefix>...__*.sql`) to avoid cross-product collisions; place repeatable views (incl. GMR alias views) under `sql_deployment/Repeatable/`.
13. **Sandbox first**: developers work in `sandbox-<name>` DAB targets writing to dev catalogs.
14. **Metric parity / reconciliation**:
    - Dataworks tests per migrated entity: row-count and `SUM(measure)` parity vs Synapse PRiME for the last full month.
    - **GMR-driven parity**: for each base measure registered in GMR against the legacy gold object, run the same aggregation on the new gold object and compare within tolerance. Run this as the canonical metric-parity test, not a hand-picked column list.
    - Dual-load until two consecutive refresh cycles match within tolerance, then cut over the GMR-pointed dataset and decommission the corresponding `PRES_<DATASET>_DEV` views.
15. **Schema drift**: rely on Dataworks drift detection at HLP; surface via CI test failures.

---

## 9. Dependencies

- **Data Integration**: `_ogr_azr_syd.<source>.*` populated (incl. `_ogr_azr_syd.midas.*`), CDF on, `rtlh_row_hash` / `rtlh_ingestion_time` present, refresh SLA per source documented; communication of any hash-column redefinition. Confirmation that big-fact sources are kept current.
- **MiDAS (Power Apps)**: continues as the MDE front-end; no change required for OGR migration.
- **GMR (Power Apps)**: base-measure / metric registry available with a documented binding to legacy `PRES_<DATASET>_DEV` columns so the migration can drive impact analysis and parity tests; owner agreed for any rename.
- **Dataworks 2.0+**: `PersistedCDF`, `FullMerge`, `DeltaLoadUsingCT`, ODW SK helpers, schema-drift detection, Key Vault integration, DAB + Flyway packaging; version pinned per OGR project from Artifactory.
- **OCDM model in Erwin**: published, with BK columns, SCD type per attribute, and a stable change-management process.
- **ODP Instance ID** allocated for the OGR ODW instance.
- **Unity Catalog**: `prime_silver`, `odw_<site>`, `prime_gold` catalogs/schemas with OGR group permissions and `sandbox_<dev>_*` schemas.
- **CI/CD**: Azure DevOps pipelines under `devops_pipelines/` for Dataworks (DAB + Flyway), replacing the DACPAC flow in PRiME.CICD.DW.yml.

---

## 10. Reference example — GTUM (HME Group Time Usage Model)

Legacy chain (Synapse, MiDAS-orchestrated, GMR-bound):

```
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DOWNTIMEAGGBYDAY
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DIMCAUSE
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DIMCAUSECATEGORY
   ↓ HLP_DEV.vw_hstg/...dim_FixedPlantTimeUsageModel + rep_/vw_cdc_ stack (HLP CDC, MiDAS-orchestrated)
HLP_DEV.dim_*, HLP_DEV.fact_HMETimeUsageEvent
   ↓ PRES_DEV.fact_HMETimeUsageEvent (OCDM, partitioned by source+year, last-2-qtrs daily)
PRES_HME_DEV.fact_HMEGroupTimeUsageEvent (GROUP BY group-event keys)
   ↓ DS Tabular / Power BI (GMR base measures + metrics; dataset IR last 2 quarters daily)
```

Target chain — Dataworks data product `data_products/ogr_hme/`, ODW instance `odw_<site>`:

| # | Asset (in `ogr_hme/`) | Tooling | UC location | Replaces |
|---|---|---|---|---|
| 1 | *(out of scope, DI)* | — | `_ogr_azr_syd.udltimeutilisation.*` (and `_ogr_azr_syd.midas.*` if MDE involved) | UDL Extractor + UDL mocks + `rep_*` / `vw_cdc_*` |
| 2 | `notebooks/hlp/hlp_hme_time_usage_model` + `config/sql/hlp_hme_time_usage_model.sql` + `sql_deployment/V001/V0001_3000000002__ogr_hme_odw_dims.sql` | `PersistedCDF` + Flyway | `prime_silver.hlp_hme.dim_time_usage_model` | HLP_DEV.vw_HSTG_UDLTIMEUTILISATION_dim_FixedPlantTimeUsageModel.sql + HLP CDC dim persistence |
| 3 | `notebooks/hlp/hlp_hme_time_usage_event` (+ config SQL) | `PersistedCDF`; incremental on `event_month` | `prime_silver.hlp_hme.fact_time_usage_event` | `HLP_DEV.vw_fact_HMETimeUsageEvent_*` + PRES partition refresh |
| 4 | `notebooks/odw/dim_hme_time_usage_model` | `FullMerge` SCD2 | `odw_<site>.dim_hme_time_usage_model` <br/>`hme_time_usage_model_sk = xxhash64(concat_ws('\|', sys_instance_id, tum5_code_bk, tum6_local_code_bk, tum7_local_code_bk))` | OCDM dim per Erwin |
| 5 | `notebooks/odw/fact_hme_time_usage_event` | Incremental MERGE; partition by `record_event_start_date_sk` month + `sys_instance_id` | `odw_<site>.fact_hme_time_usage_event` | `PRES_DEV.fact_HMETimeUsageEvent` / `m_fact_*` + last-2-qtrs partition refresh |
| 6 | `sql_deployment/Repeatable/R__gold_hme_views.sql` | Flyway repeatable view (+ legacy-name alias columns/view for GMR) | `prime_gold.hme.fact_hme_group_time_usage_event` (alias view `fact_HMEGroupTimeUsageEvent` and alias columns `cGroupEventTUM`, `sGroupEventTUM`) | PRES_HME_DEV.fact_HMEGroupTimeUsageEvent.sql |
| 7 | GMR base measures `cGroupEventTUM`, `sGroupEventTUM` + metric `HME Group Event TUM` | Re-pointed via Tabular (Direct Lake / DirectQuery, or Import + IR) | semantic layer | Same GMR registrations, gold-pointed; no re-registration thanks to alias columns |
| 8 | `resources/jobs/ogr_hme.job.yml` | DAB | — | MiDAS dataflow metadata |
| 9 | `devops_pipelines/ogr_hme.azure-pipelines.yml` | Azure DevOps | — | PRiME.CICD.DW.yml |

DAB job graph (replaces MiDAS dataflow metadata): `hlp_hme_time_usage_model` ∥ `hlp_hme_time_usage_event` → `dim_hme_time_usage_model` → `fact_hme_time_usage_event` → gold view refresh → reconciliation tests.

Reconciliation tests in the bundle:
- Row-count and `SUM(c_group_event_tum)`, `SUM(s_group_event_tum)` parity per `group_event_start_date_sk` for last full month vs Synapse `PRES_HME_DEV.fact_HMEGroupTimeUsageEvent`.
- SK uniqueness on every ODW dim/fact.
- **GMR-driven parity**: every base measure registered against `fact_HMEGroupTimeUsageEvent` re-evaluated on the new gold table; tolerance per measure.
- CRI cases: `sys_first_seen_datetime_utc` parity with the legacy Work-Day-5 snapshot.
- Two consecutive matching refresh cycles → cut over → decommission Synapse `PRES_HME_DEV` HME views.

---

## 11. Acceptance-criteria coverage

| Criterion | Section |
|---|---|
| Transformation, not lift-and-shift | §3, §6 |
| Dataworks 2.0 (DAB + Flyway + reusable notebooks + ODW) confirmed as standard | §3, §4, §5, §6 |
| End-to-end flow incl. ingestion → ODW (OCDM) → star → GMR | §3 diagram + §3.1 tier mapping |
| Dataworks patterns for ingest consumption + OCDM conformance | §6.1–6.4 |
| Relationship Dataworks ↔ OCDM (Erwin) ↔ ODW ↔ star ↔ GMR (base measures + metrics) | §7 |
| Guidelines for SQL re-implementation + metric parity / reconciliation (GMR-driven) | §8 |
| Legacy CDC tiers explicitly mapped | §1.1 + §3.1 |
| MiDAS role correctly characterised (Power Apps; MDE retained, orchestration metadata replaced) | §1, §2.1, §3, §3.1, §6.1, §7 |
| GMR role correctly characterised (Power Apps; base measures + metrics; gold-column contract) | §1, §2.2, §3, §6.3, §7, §8 |
| Project layout aligned to official Dataworks structure (`.rtlh_dataworks/`, `config/`, `core/`, `devops_pipelines/`, `notebooks/`, `resources/jobs|pipelines/`, `sql_deployment/Repeatable|V001/`, `project.schema.toml`, `databricks.yml`) | §5 |
| Flyway versioning convention (project-prefixed) | §5, §8 |
| Dependencies (Dataworks 2.0, OCDM/Erwin, Instance ID, DI source contract, MiDAS, GMR) | §9 |
| GTUM reference example | §10 |
| Documented for tech-lead sign-off | this doc |

# OGR – Dataworks Transformation Approach (Final, ODW v2 + correct GMR + correct Dataworks project structure)

Updates applied:
1. **GMR** correctly characterised — Power Apps app for **defining Power BI measures** with **Base Measure** and **Metric** layers (not just "metrics in the semantic layer").
2. **Project layout** rewritten to the official Dataworks Data Product structure (`.rtlh_dataworks/`, `config/`, `core/`, `devops_pipelines/`, `notebooks/`, `resources/{jobs,pipelines}/`, `sql_deployment/{Repeatable,V001}/`, `project.schema.toml`, `databricks.yml`).

---

## 1. Current PRiME / Synapse DW (recap)

Layered Synapse SQL solution under 1. DEV:

- **UDL (mocked HSTG_V / DAL_V)** → **HLP_DEV** (per-source `vw_hstg`, `vw_dim`, `vw_fact` + custom CDC via `rep_*` + `vw_cdc_*` + `cdc.*` + MiDAS-driven `mde` views) → **PRES_DEV** (OCDM-aligned `dim_*`, `fact_*`, `m_fact_*`) → **PRES_<DATASET>_DEV** (HME / RAIL / EEDRILL / EEDRILLCOST / AM / PROCESS / PROCESSASSURANCE / GLOBAL_METRICS star schemas) → **DS Tabular models** (consumed by Power BI; measures defined in **GMR**).
- **MiDAS** (Power Apps): dataflow / dataflow-dependency metadata + manual data entry.
- **GMR – Global Metric Registry** (Power Apps): defines Power BI **Base Measures** and **Metrics** that the DS Tabular models materialise. Two-layer model:
  - **Base Measure layer** — atomic building blocks (e.g. `sum(c_group_event_tum)` per dimensionality).
  - **Metric layer** — business KPIs composed from Base Measures (ratios, per-shift, per-asset, etc.).
- Refresh = Synapse Data Flows orchestrated by MiDAS metadata.
- Deploy = DACPAC compare → SQL script → manual SSMS run, with folder staging via Stage SQL Objects.ps1.

### 1.1 Legacy CDC / refresh tiers (today)

| # | Tier | Where | What it does today | Limits |
|---|---|---|---|---|
| 1 | **UDL Extractor** | Source → UDL `DAL` / `HSTG` | Full load most tables; for very large tables (Cycle / Event from ORDW & MMRS) only last 3–6 months refreshed daily. | Window-based, not true CDC. |
| 2 | **HLP CDC** | UDL → `HLP_DEV.dim_*` and CRI `HLP_DEV.fact_*` | Full compare on dims (so SKs persist for facts); partial 3-month compare on CRI fact tables to capture Work-Day-5 snapshot timestamp. | Bespoke `rep_*` + `vw_cdc_*` machinery. |
| 3 | **PRES Partition Refresh** | HLP → `PRES_DEV.fact_*` | Source-system or source-system+year partitions; daily refresh of last 2 quarters for big facts (HME Cycle/Event). | Manual partition load procs. |
| 4 | **Dataset Incremental Refresh** | PRES → Tabular / Power BI | HME Cycle/Event refresh last 2 quarters daily; everything else fully refreshed. | Refresh time bottleneck. |

All four tiers are driven by metadata maintained in **MiDAS (Power Apps)**, not Excel.

Pain to remove: bespoke HLP CDC, two parallel "CDC vs partition refresh" patterns, IDENTITY-based SKs (RemoveIdentityColumn.sql), `1. DEV / 2. TEST / 3. PROD` folder staging, MiDAS-driven dataflow metadata as the orchestration source of truth, manual partition loaders, dataset-level incremental refresh as the only way to keep BI fast.

---

## 2. Source contract from Data Integration (out of OGR scope)

- Catalog **`_ogr_azr_syd`**, schema per source system; same `DAL_V.*` / `HSTG_V.*` entities used today.
- Each table is **Delta with CDF enabled** plus:
  - `rtlh_row_hash` — hash of business columns,
  - `rtlh_ingestion_time` — last ingest timestamp.
- Refresh MERGE keyed on `rtlh_row_hash`: unchanged → no-op; changed → old hash deleted + new hash inserted; dropped → deleted; new → inserted.

This single tier replaces the **UDL Extractor** + **HLP CDC** combo. CDF consumed via Dataworks **`PersistedCDF`**; updates appear as `(delete, insert)` pairs at the same `_commit_version`; downstream joins use **business keys**, not `rtlh_row_hash`; hard deletes propagate (handled as `is_deleted` in ODW dims).

### 2.1 MiDAS in the new world

- **Manual data entry** — still owned by MiDAS. MDE tables ingested into `_ogr_azr_syd.midas.*` by Data Integration on the same CDF contract. OGR consumes them via `prime_silver.hlp_midas.*` (replaces `HLP_DEV.mde.*`).
- **Orchestration metadata** — **no longer used by OGR**. Job graph, schedules, dependencies, environments live in the Dataworks DAB (`databricks.yml` + `resources/jobs/*.yml`).

### 2.2 GMR in the new world

- **GMR remains the source of truth for Power BI Base Measures and Metrics.** The Tabular model continues to materialise the GMR catalogue.
- What changes: the Tabular model's underlying tables move from Synapse `PRES_<DATASET>_DEV` to Databricks `prime_gold.<dataset>` (via SQL Warehouse). GMR Base Measures may need column-name remapping where ODW snake_case is exposed.
- The Gold layer is contractually shaped to match the columns Base Measures expect — this is enforced by **GMR-derived parity tests** in the Dataworks bundle (see §6, §8).

---

## 3. Target architecture on Databricks + Dataworks 2.0 (ODW)

```mermaid
flowchart LR
    subgraph DI["Data Integration (out of OGR scope)"]
        SRC[Source systems]
        MIDAS["MiDAS (Power Apps)<br/>Manual Data Entry"]
        ING["_ogr_azr_syd.&lt;source&gt;.*<br/>incl. _ogr_azr_syd.midas.*<br/>Delta + CDF<br/>rtlh_row_hash / rtlh_ingestion_time"]
        SRC --> ING
        MIDAS --> ING
    end

    subgraph OGR["OGR Dataworks Data Products (one per dataset)"]
      direction TB
      subgraph HLP["prime_silver.hlp_&lt;source&gt;"]
        H1[Per-source shaping<br/>PersistedCDF / FullMerge<br/>incl. hlp_midas.* for MDE]
      end
      subgraph ODW["odw_&lt;site&gt; (OCDM on Databricks)"]
        O1["dim_* (snake_case)<br/>SK = xxhash64(sys_instance_id | bk_cols)"]
        O2["fact_* / m_fact_*<br/>partition by event-month"]
      end
      subgraph GOLD["prime_gold.&lt;dataset&gt; (Star schema)"]
        G1[dim_* / fact_* per OGR dataset<br/>+ legacy-name aliases]
      end
    end

    subgraph Sem["Semantic + metric layer"]
        TAB[DS Tabular models on Databricks SQL Warehouse]
        GMR["GMR (Power Apps)<br/>Base Measures → Metrics"]
        PBI[Power BI reports]
        TAB --> PBI
        GMR -.defines.-> TAB
    end

    ING --> HLP --> ODW --> GOLD --> TAB

    Erwin[(OCDM<br/>Erwin)] -.contract.-> ODW
    InstanceReg[(ODP Instance ID registry)] -.sys_instance_id.-> ODW
    Flyway[Flyway CLI<br/>sql_deployment/V*** + Repeatable/] -.deploys.-> HLP
    Flyway -.deploys.-> ODW
    Flyway -.deploys.-> GOLD
    DAB[DAB CLI<br/>databricks.yml + resources/jobs] -.orchestrates.-> HLP
    DAB -.orchestrates.-> ODW
    DAB -.orchestrates.-> GOLD
    KV[Azure Key Vault] -.auth.-> DAB
```

### 3.1 Tier-by-tier mapping (legacy → target)

| Legacy tier | Target on Databricks |
|---|---|
| UDL Extractor (full + 3/6-month windows) | DI's CDF-enabled `_ogr_azr_syd.<src>.*`. |
| HLP CDC – dimensions (full compare, persist SK) | `prime_silver.hlp_<src>` (`PersistedCDF`) → `odw_<site>.dim_*` with **deterministic hash SK**. |
| HLP CDC – CRI fact tables (Work-Day-5 snapshot) | `prime_silver.hlp_cri.fact_*` consuming CDF; `sys_first_seen_datetime_utc` captures snapshot timestamp. |
| PRES Partition Refresh (last 2 quarters daily) | `odw_<site>.fact_*` partitioned by `event_month` (+ `sys_instance_id` for big facts); incremental MERGE driven by CDF bookmark; `m_fact_*` = last 6 months hot. |
| Dataset Incremental Refresh (Power BI) | Direct Lake / DirectQuery against gold; or Import + Power BI IR aligned to `event_month`. |
| MiDAS dataflow / dependency metadata | DAB `databricks.yml` + `resources/jobs/*.yml` in Git. |
| MiDAS manual data entry | Unchanged — `_ogr_azr_syd.midas.*` → `prime_silver.hlp_midas.*`. |
| GMR Base Measures / Metrics | Unchanged — GMR still owns Power BI measure definitions; Tabular re-points to `prime_gold.<dataset>`. |

---

## 4. ODW conventions adopted (mandatory)

- **Schema**: `odw_<site_code>` (e.g. `odw_wei`).
- **Tables / columns**: snake_case (`fact_fixed_plant_time_usage_event`, `site_sk`).
- **Surrogate keys** (Fact & Dim):
  - `xxhash64(concat_ws('|', sys_instance_id, <bk_col_1>, <bk_col_2>, ...)) AS <entity>_sk`
  - Removes IDENTITY plumbing; supports **fact-first ETL with stub rows** for late-arriving dims.
- **`sys_instance_id`** present on every ODW table and included in every SK hash.
- **Datetime**:
  - System: `timestamp` UTC — `sys_created_datetime_utc`, `sys_updated_datetime_utc`.
  - Business: `timestamp_ntz` local — `record_event_datetime`, etc.
  - Date SKs: `date` local — `record_event_start_date_sk`, etc.
  - `sys_timezone_name` (IANA) on every ODW table.
- **Soft-delete / SCD**: `is_deleted` + SCD2 close-row pattern; default Type-1 unless Erwin marks attribute historised.
- **CRI snapshot semantics**: `sys_first_seen_datetime_utc` on relevant ODW facts.

Legacy `OMD_*` / `CDC_*` columns kept only in HLP for traceability, not propagated to ODW.

---

## 5. Standard Dataworks Data Product layout (per OGR dataset)

Aligned with the official Dataworks project structure — created via `dataworks create` (DAB custom template):

```
data_products/ogr_hme/
│── .rtlh_dataworks/                      # Reserved Dataworks reusable components (PersistedCDF, FullMerge, ODW SK helpers, etc.)
│── config/
│   ├── json/                             # JSON config (e.g. DLT pipeline config, ingestion source list, ocdm mapping)
│   │   ├── ingestion_sources.json        # HLP source list + bk columns per entity
│   │   └── ocdm_mapping.json             # OCDM/Erwin-derived mapping (entity → bk cols → SK)
│   └── sql/                              # Source-extraction / transformation SQL referenced by notebooks
│       ├── hlp/hlp_hme_time_usage_model.sql
│       ├── hlp/hlp_hme_time_usage_event.sql
│       ├── odw/dim_hme_time_usage_model.sql
│       └── odw/fact_hme_time_usage_event.sql
│── core/                                 # Project-local reusable Python / helpers (e.g. CRI snapshot helper, GMR-parity test helper)
│── devops_pipelines/
│   └── ogr_hme.ci-cd.yml                 # Azure DevOps pipeline: dataworks build → flyway migrate → DAB deploy per target
│── notebooks/                            # Dev / exploratory notebooks (not part of core run-time)
│── resources/
│   ├── jobs/
│   │   └── ogr_hme.job.yml               # DAB job: HLP → ODW → Gold task graph + schedule (replaces MiDAS dataflow metadata)
│   └── pipelines/                        # DLT pipelines (only if any layer is implemented as DLT)
│── sql_deployment/                       # Flyway-managed DDL
│   ├── V0001_<projprefix>0000000001__create_schemas.sql
│   ├── V0001_<projprefix>0000000010__odw_<site>_dims.sql
│   ├── V0001_<projprefix>0000000020__odw_<site>_facts.sql
│   └── Repeatable/
│       ├── R__gold_hme_views.sql                       # incl. legacy-name aliases for GMR back-compat
│       └── R__gold_hme_gmr_contract_views.sql          # views shaped to GMR Base Measures
│── project.schema.toml                   # Schema definitions for the data product
│── databricks.yml                        # DAB root: targets (dev, test, prod, sandbox-<dev>), variables, includes
```

Notes:
- **Environments are DAB targets**, not folders → eliminates the legacy `1. DEV / 2. TEST / 3. PROD` staging.
- **Flyway versioning** uses the project-prefixed convention (`V0001_<projprefix>NNNNNNNNN__*.sql`) so multiple OGR data products can share the same Flyway repo without version collisions.
- **`Repeatable/`** holds the gold-layer views (which are pure projections / aliases) and the **GMR-contract views** that pin the column shape Base Measures expect.
- **`.rtlh_dataworks/`** is reserved — OGR teams must not modify it.

One Data Product per OGR dataset (`ogr_hme`, `ogr_rail`, `ogr_eedrill`, `ogr_eedrillcost`, `ogr_am`, `ogr_process`, `ogr_processassurance`, `ogr_global_metrics`).

---

## 6. Patterns per layer

### 6.1 HLP — per-source shaping

- Notebooks under `notebooks/hlp/` (or `core/` if reusable) parameterised by SQL files in `config/sql/hlp/`.
- Use the reusable `PersistedCDF` template from `.rtlh_dataworks/`.
- Bookmark = last `_commit_version` per (source, target). `(delete, insert)` pairs handled by `MERGE` on the deterministic hash SK.
- One HLP model per current `HLP_DEV` view → 1:1 reconciliation.
- **MDE handling**: `prime_silver.hlp_midas.*` models consume `_ogr_azr_syd.midas.*` and replicate the harmonisation done today by `HLP_DEV.mde.*`.
- For CRI: capture `sys_first_seen_datetime_utc = least(existing, current_ingest_time)` (helper in `core/`).

### 6.2 ODW — `odw_<site>` (OCDM on Databricks)

- One Dataworks notebook per Erwin entity → `odw_<site>.dim_*` / `fact_*`. SQL bodies in `config/sql/odw/`.
- SK = `xxhash64(concat_ws('|', sys_instance_id, <bk_cols>))` — stable across full reloads.
- SCD2 via `FullMerge` on Erwin-flagged historised attributes; default Type-1.
- Hard deletes → `is_deleted = true` + close current SCD2 row.
- Facts partitioned by event-month; big facts also by `sys_instance_id`.
- `m_fact_*` = last-6-months hot + `OPTIMIZE … ZORDER BY` on hot dim FKs.

### 6.3 Gold — per-dataset star schema (replaces `PRES_<DATASET>_DEV`)

- Implemented as **Flyway repeatable views** in `sql_deployment/Repeatable/R__gold_<ds>_views.sql` (no daily compute — they're projections over ODW).
- Two view "skins" per object where useful:
  - **Canonical snake_case view** matching ODW conventions.
  - **Legacy PascalCase alias view** during cut-over for GMR back-compat (e.g. `fact_HMEGroupTimeUsageEvent` → `fact_hme_group_time_usage_event`).
- Role-playing splits implemented as separate views (e.g. `fact_hme_base_time_usage_event`, `fact_hme_group_time_usage_event`, `fact_hme_record_time_usage_event` from one ODW fact).

### 6.4 GMR / semantic layer

- **No change in metric ownership** — Base Measures and Metrics continue to be defined and managed in **GMR (Power Apps)**.
- DS Tabular models point at `prime_gold.<dataset>` via Databricks SQL Warehouse.
- For each migrated table, the bundle ships a **GMR-contract view** (`Repeatable/R__gold_<ds>_gmr_contract_views.sql`) that exposes exactly the columns the affected Base Measures reference, preventing accidental breakage as ODW evolves.
- Power BI dataset incremental refresh, where still needed, is configured against the gold table's `event_month` column instead of being model-only.

### 6.5 Orchestration, environments, security

- **DAB job per data product** in `resources/jobs/<product>.job.yml` wires HLP → ODW → Gold task graph and owns dependencies + schedules (replacing MiDAS dataflow metadata).
- Targets in `databricks.yml`: `dev`, `test`, `prod`, `sandbox-<dev>`.
- Auth: Azure Key Vault via Dataworks integration.
- CI/CD in `devops_pipelines/<product>.ci-cd.yml`: `dataworks build` → `flyway migrate` (consumes `sql_deployment/`) → `databricks bundle deploy --target <env>` — replacing the DACPAC + GO-stripping workaround in PRIME.CICD.DW-uses local server.yml.

---

## 7. Relationship between artefacts

| Artefact | Owns | Drives |
|---|---|---|
| **Dataworks Data Product (DAB + Flyway + reusable notebooks)** | All OGR code, DDL, **and orchestration / dependencies** (the role MiDAS dataflow metadata played). | Build order, tests, multi-env deploy, sandboxes. |
| **OCDM model in Erwin** | Logical contract for `odw_<site>.*`. | Entity / attribute names, datatypes, BK columns, SCD type, conformed dims. |
| **ODP Instance ID registry** | `sys_instance_id` per ODW instance. | SK hashing salt; cross-site uniqueness. |
| **MiDAS (Power Apps)** | Manual data entry only in the new world. | Source of `_ogr_azr_syd.midas.*` → `prime_silver.hlp_midas.*`. |
| **Star schema (`prime_gold.<ds>`)** | Physical analytical model per OGR dataset. | Query shape, role-playing dims, legacy-name aliases, GMR-contract views. |
| **GMR (Power Apps)** | Power BI **Base Measures** and **Metrics**. | DS Tabular measure definitions; parity tests in the Dataworks bundle. |

---

## 8. Reimplementation guidelines

1. **Out of scope for OGR**: UDL mocked views, `[HLP_DEV].[sp_UDLVWDesc]`, all `rep_*` / `vw_cdc_*` / `cdc.*` objects, `1. DEV / 2. TEST / 3. PROD` staging, the UDL Extractor, manual partition load procs, MiDAS dataflow metadata as orchestrator.
2. **MiDAS treatment**: keep MDE; drop the orchestration metadata role. Map every `HLP_DEV.mde.*` view to a `prime_silver.hlp_midas.*` Dataworks model.
3. **GMR treatment**: keep Base Measures & Metrics in GMR; update only the table/column bindings to the new gold layer. Use the GMR-contract repeatable views as the stable interface.
4. **Per legacy SQL file**: classify into HLP / ODW / gold and create the matching Dataworks asset (notebook + SQL in `config/sql/` + Flyway DDL in `sql_deployment/`).
5. **Naming**: ODW objects in snake_case. Legacy PascalCase survives only as gold-layer alias views during cut-over.
6. **Surrogate keys**: ODW v2 hash with `sys_instance_id` salt. No IDENTITY columns.
7. **CDF consumption**: `PersistedCDF`; persist `_commit_version` bookmark; treat `(delete, insert)` pairs as logical update; never join on `rtlh_row_hash`.
8. **CRI snapshot semantics**: capture `sys_first_seen_datetime_utc` in HLP.
9. **Big facts (HME Cycle/Event, MMRS Event)**: partition by `event_month` (+ `sys_instance_id`); incremental MERGE replaces "last 2 quarters daily" partition refresh.
10. **SCD**: default Type-1; Type-2 only where Erwin marks attribute historised; deletes → `is_deleted = true` + close current row.
11. **Datetime**: enforce ODW datatype split; always populate `sys_timezone_name`.
12. **Sandbox first**: developers work in `sandbox-<name>` DAB targets writing to dev catalogs.
13. **Flyway versioning**: use the project-prefixed convention `V0001_<projprefix>NNNNNNNNN__*.sql` to avoid cross-project collisions.
14. **Metric parity / reconciliation** (three layers):
    - **Table**: row-count and `SUM(measure)` parity per migrated entity vs the live Synapse PRiME DW for the last full month.
    - **Base Measure** (GMR): for each affected GMR Base Measure, evaluate it against both old and new gold layer for the same dim slice; require parity within tolerance.
    - **Metric** (GMR): for each affected GMR Metric, evaluate the composed measure end-to-end in Power BI for the standard report pages.
    - Dual-load until two consecutive refresh cycles match → cut over the GMR dataset and decommission the corresponding `PRES_<DATASET>_DEV` views.
15. **Schema drift**: rely on Dataworks drift detection at HLP; surface via CI test failures.

---

## 9. Dependencies

- **Data Integration**: `_ogr_azr_syd.<source>.*` populated (incl. `_ogr_azr_syd.midas.*` for MDE), CDF on, `rtlh_row_hash` / `rtlh_ingestion_time` present, refresh SLA per source documented; communication of any hash-column redefinition. Big-fact sources kept current (replaces UDL Extractor 3/6-month windows).
- **MiDAS (Power Apps)**: continues as the MDE front-end; no change required for OGR migration.
- **GMR (Power Apps)**: continues to own Base Measures and Metrics; mapping update from PRES tables → gold tables required as part of cut-over per dataset.
- **Dataworks 2.0+**: `PersistedCDF`, `FullMerge`, `DeltaLoadUsingCT`, ODW SK helpers, schema-drift detection, Key Vault integration, DAB + Flyway packaging; version pinned per OGR data product from Artifactory.
- **OCDM model in Erwin**: published, with BK columns, SCD type per attribute, and a stable change-management process.
- **ODP Instance ID** allocated for the OGR ODW instance.
- **Unity Catalog**: `prime_silver`, `odw_<site>`, `prime_gold` catalogs/schemas with OGR group permissions and `sandbox_<dev>_*` schemas.
- **DS Tabular models**: re-point to Databricks SQL Warehouse; decision on Direct Lake / DirectQuery vs Import + IR per dataset.
- **CI/CD**: Azure DevOps pipelines re-templated for Dataworks (DAB + Flyway), replacing the DACPAC flow in PRiME.CICD.DW.yml.

---

## 10. Reference example — GTUM (HME Group Time Usage Model)

Legacy chain (Synapse, MiDAS-orchestrated, GMR-defined measures):

```
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DOWNTIMEAGGBYDAY
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DIMCAUSE
HSTG_V.VW_HSTG_UDLTIMEUTILISATION_DBO_DIMCAUSECATEGORY
   ↓ HLP_DEV.vw_hstg/...dim_FixedPlantTimeUsageModel + rep_/vw_cdc_ stack (HLP CDC, full compare; MiDAS-orchestrated)
HLP_DEV.dim_*, HLP_DEV.fact_HMETimeUsageEvent
   ↓ PRES_DEV.fact_HMETimeUsageEvent (OCDM, partitioned by source+year, last-2-qtrs daily)
PRES_HME_DEV.fact_HMEGroupTimeUsageEvent (GROUP BY group-event keys)
   ↓ DS Tabular (measures defined in GMR: Base Measures e.g. sum_c_group_event_tum, Metrics e.g. HME Group Event TUM)
   ↓ Power BI (dataset IR last 2 quarters daily)
```

Target chain — Dataworks Data Product `data_products/ogr_hme/`, ODW instance `odw_<site>`:

| # | Dataworks asset | Path | UC location | Replaces |
|---|---|---|---|---|
| 1 | *(out of scope, DI)* | — | `_ogr_azr_syd.udltimeutilisation.*` (and `_ogr_azr_syd.midas.*` if MDE involved) | UDL Extractor + UDL mocks + `rep_*` / `vw_cdc_*` |
| 2 | `hlp_hme_time_usage_model` | `notebooks/hlp/...` + `config/sql/hlp/hlp_hme_time_usage_model.sql` + DDL in `sql_deployment/V0001_<prefix>0000000010__hlp_hme.sql` | `prime_silver.hlp_hme.dim_time_usage_model` | HLP_DEV.vw_HSTG_UDLTIMEUTILISATION_dim_FixedPlantTimeUsageModel.sql + HLP CDC dim persistence |
| 3 | `hlp_hme_time_usage_event` | `notebooks/hlp/...` + `config/sql/hlp/...` + DDL in `sql_deployment/...` | `prime_silver.hlp_hme.fact_time_usage_event` | `HLP_DEV.vw_fact_HMETimeUsageEvent_*` + PRES partition refresh |
| 4 | `dim_hme_time_usage_model` | `notebooks/odw/...` + `config/sql/odw/...` + DDL in `sql_deployment/...__odw_<site>_dims.sql` | `odw_<site>.dim_hme_time_usage_model` <br/>`hme_time_usage_model_sk = xxhash64(concat_ws('\|', sys_instance_id, tum5_code_bk, tum6_local_code_bk, tum7_local_code_bk))` | OCDM dim per Erwin |
| 5 | `fact_hme_time_usage_event` | `notebooks/odw/...` + `config/sql/odw/...` + DDL in `sql_deployment/...__odw_<site>_facts.sql` (partition by `record_event_start_date_sk` month + `sys_instance_id`) | `odw_<site>.fact_hme_time_usage_event` | `PRES_DEV.fact_HMETimeUsageEvent` / `m_fact_*` + last-2-qtrs partition refresh |
| 6 | `gold_hme_views` | `sql_deployment/Repeatable/R__gold_hme_views.sql` (incl. `fact_HMEGroupTimeUsageEvent` alias) + `Repeatable/R__gold_hme_gmr_contract_views.sql` | `prime_gold.hme.fact_hme_group_time_usage_event` | PRES_HME_DEV.fact_HMEGroupTimeUsageEvent.sql |
| 7 | DAB job graph | `resources/jobs/ogr_hme.job.yml` | n/a | MiDAS dataflow metadata for HME |
| 8 | GMR Base Measures (`sum_c_group_event_tum`, `sum_s_group_event_tum`, …) and Metric `HME Group Event TUM` | GMR (Power Apps) — **rebound** to gold tables | semantic layer | Same Base Measures / Metric, gold-pointed |

Reconciliation tests in the bundle (run by CI and pre-cut-over):
- **Table parity**: row-count and `SUM(c_group_event_tum)`, `SUM(s_group_event_tum)` per `group_event_start_date_sk` for last full month vs Synapse `PRES_HME_DEV.fact_HMEGroupTimeUsageEvent`.
- **SK uniqueness** on every ODW dim/fact.
- **CRI cases**: `sys_first_seen_datetime_utc` parity with the legacy Work-Day-5 snapshot.
- **GMR Base Measure parity**: each HME Base Measure evaluated against old vs new gold for the same dim slice.
- **GMR Metric parity** in Power BI for the standard HME Time Usage report pages.
- Two consecutive matching refresh cycles → cut over → decommission Synapse `PRES_HME_DEV` HME views.

---

## 11. Acceptance-criteria coverage

| Criterion | Section |
|---|---|
| Transformation, not lift-and-shift | §3, §6 |
| Dataworks 2.0 (DAB + Flyway + reusable notebooks + ODW) confirmed as standard | §3, §4, §5, §6 |
| Project layout matches official Dataworks Data Product structure | §5 |
| End-to-end flow ingestion → ODW (OCDM) → star → GMR Tabular → Power BI | §3 diagram + §3.1 tier mapping |
| Dataworks patterns for ingest consumption + OCDM conformance | §6.1–6.4 |
| Relationship Dataworks ↔ OCDM (Erwin) ↔ ODW ↔ star ↔ GMR (Base Measures + Metrics) | §7 |
| Guidelines for SQL re-implementation + three-layer parity (table / Base Measure / Metric) | §8 |
| Legacy CDC tiers explicitly mapped | §1.1 + §3.1 |
| MiDAS role correctly characterised (Power Apps; MDE retained, orchestration metadata replaced) | §1, §2.1, §3.1, §6.1, §7 |
| GMR role correctly characterised (Power Apps; Base Measure + Metric layers; rebound to gold) | §1, §2.2, §6.4, §7 |
| Dependencies (Dataworks 2.0, OCDM/Erwin, Instance ID, DI source contract, MiDAS, GMR) | §9 |
| GTUM reference example with full project-structure mapping | §10 |
| Documented for tech-lead sign-off | this doc |

Ran `view_server_info` 
Completed with input: {}

Ran `get_sql_formatting_conventions` 
Completed with input: {}

Ran `get_hash_key_conventions` 
Completed with input: {}

Ran `get_null_handling_conventions` 
Completed with input: {}

Ran `get_dimension_join_conventions` 
Completed with input: {}

Ran `get_timezone_conventions` 
Completed with input: {}

Ran `get_system_columns_conventions` 
Completed with input: {}

Ran `get_delta_load_conventions` 
Completed with input: {}

Ran `get_file_naming_conventions` 
Completed with input: {}

Task completed successfully
Completed with input: {
  "keyword": "transformation notebook",
  "query": "reusable transformation notebooks FullLoad DeltaLoad strategies"
}

Read [](file:///c%3A/Users/byambadorjme/AppData/Roaming/Code/User/workspaceStorage/03b859180bddca37e35c6bc2b1d02c95/GitHub.copilot-chat/chat-session-resources/6a8c6593-5883-4495-9ca3-53c0215eed7b/toolu_vrtx_01BQMcF8HAx2PngPaPce6Q64__vscode-1777360151028/content.json)

Pulled the authoritative conventions from the Dataworks MCP. Several specifics in the previous playbook were wrong (e.g. `_sk` vs `_hk`, `sys_instance_id` salting, snake_case source columns, target schema naming). Regenerated below using the official conventions.

---

# OGR – PRiME → Databricks Migration Playbook (Practical Guide, MCP-aligned)

Companion to the high-level approach doc. For engineers doing the actual conversion: **given a Synapse SQL object, what Dataworks asset do I produce, and how?**

Scope: **HLP layer + ODM transformations only**. Out of scope: ingestion (Data Integration), gold / star schema, Tabular / Power BI / GMR rebinding.

> Authoritative source for all conventions in §2, §6, §7 = `mcp_dataworks_*` (file naming, hash keys, null handling, dim joins, timezone, system columns, delta-load CDF). Where the Dataworks doc and the MCP conventions disagreed, MCP wins.

---

## 1. Prerequisites

- [ ] Dataworks Data Product scaffolded via `dataworks create` (DAB custom template).
- [ ] `${env_ingestion_catalog}.<source_schema>${env_schema_suffix}.<table>` exists for every UDL view your HLP_DEV objects read — confirm with Data Integration (CDF on; refresh contract documented).
- [ ] OCDM entity for the target ODM table is published in Erwin with **business keys** identified.
- [ ] Sandbox DAB target deploys cleanly.
- [ ] Dataworks package version pinned; reusable strategies visible under `.rtlh_dataworks/`.

---

## 2. Catalogue of Dataworks ODM transformation strategies (reusable notebooks)

From the RTLH Dataworks WIKI → "ODM Transformation Projects":

| # | Strategy notebook | Purpose | When to use |
|---|---|---|---|
| 1 | **ODM Full Merge** | Runs a full-load SQL and `MERGE`s into the ODM Delta table (insert / update / delete by hash key). | Dimensions and small reference tables; daily full rebuild of the source set. |
| 2 | **ODM Delta Load Using Change Data Feed** | Runs a delta-load SQL against a CDF source (`TABLE_CHANGES()` or `_cdf` table); falls back to full load on first run / empty target. | **Default for OGR facts**, since every `_ogr_azr_syd.*` source is CDF-enabled. |
| 3 | **Persisted Change Data Feed** | DLT streaming table that materialises a per-source `_cdf` history table from the source's CDF. Optional precursor for entities consumed by Pattern 2 (`_cdf`-table pattern). | When you need a persisted CDF history (audit, multi-consumer reuse) before transforming. |
| 4 | **ODW Transformation Projects** (custom) | Python/SQL transformation notebook hosted by the framework; allows multi-step transformations, parameter loading, library init. | Only when none of the strategy notebooks fit (multi-step pipelines, custom orchestration). |

Ingestion-side strategies — listed for awareness only; **not used by OGR**:
- Delta Load Using Change Tracking (SQL Server CT)
- Delta Load using Watermark with All Keys
- Ranged Reload
- (with `AutoCreateTable` + schema-drift handling)

> Rule of thumb for OGR: **dimensions → ODM Full Merge; facts → ODM Delta Load Using CDF.** Anything else → custom ODW transformation notebook.

---

## 3. Decision tree — legacy file → strategy

```
START
  ├─ PRES_<DATASET>_DEV/*           → OUT OF SCOPE here (gold layer)
  │
  ├─ PRES_DEV/dim/*                 → ODM Full Merge      (custom ODW notebook only if multi-step)
  ├─ PRES_DEV/fact/* | m_fact/*     → ODM Delta Load Using CDF
  │
  ├─ HLP_DEV/vw_hstg/*              → SQL transform body for the downstream ODM strategy (no standalone notebook)
  │
  ├─ HLP_DEV/vw_dim/* + dim/*       → ODM Full Merge
  │                                    (one .FullLoad.sql per source system OR one query that UNION ALLs sources)
  │
  ├─ HLP_DEV/vw_fact/* + fact/*     → ODM Delta Load Using CDF        (.DeltaLoad.sql)
  │   └─ Source has no CDF & has    → Custom notebook using watermark (rare for OGR)
  │      a watermark column
  │
  ├─ HLP_DEV/rep_* | vw_cdc_* | cdc.* → DELETE (CDF replaces this stack)
  ├─ HLP_DEV/mde/*                  → ODM Full Merge over _ogr_azr_syd.midas.*
  ├─ HLP_DEV/fn/*                   → port to Spark SQL (regex / window) or Spark UDF in core/
  ├─ HLP_DEV/sp/*                   → fold logic into the relevant SQL; no procedural recreate
  │
  └─ DW/Scripts/*                   → superseded by framework features (see §8)
```

---

## 4. Per-file conversion recipe

For **every** legacy `HLP_DEV.<view>.sql` and `PRES_DEV.<entity>.sql`:

### Step 1 — Locate and read the legacy SQL

Find the source under 1. DEV. Note:

1. **Sources** — every `HSTG_V.*` / `DAL_V.*` becomes `${env_ingestion_catalog}.<source_schema>${env_schema_suffix}.<table>`.
2. **Business keys** of the target — usually the natural-key columns visible in the matching `rep_*` / `dim_*` table DDL.
3. **Load type** — dim → `FullLoad`; fact → `DeltaLoad`.

### Step 2 — Pick the strategy (§3)

### Step 3 — Write the SQL file

- Path: `config/sql/{schema}.{table_name}.{FullLoad|DeltaLoad}.sql`
  - e.g. `config/sql/odm.fixed_plant_time_usage_event.DeltaLoad.sql`
  - e.g. `config/sql/odm.hme_time_usage_model.FullLoad.sql`
- Apply all the conventions in §6.
- Use environment placeholders verbatim: `${env_ingestion_catalog}`, `${env_schema_suffix}`, `${workspace.file_path}`.
- For `DeltaLoad`, use one of the two CDF patterns in §7.

### Step 4 — Add config JSON

`config/json/{schema}.{entity}.config.json` — strategy parameters (target table, business keys, hash key column, partition columns, etc.).

### Step 5 — Add Flyway DDL

`sql_deployment/V{version}/V{version}_{timestamp}__Create_Table_{schema}.{table_name}.sql` (timestamp `YYYYMMDDHHmmss`). The DDL must include the auto-generated `_hk` columns and the system columns (§6.4).

### Step 6 — Wire the job

Add a task to `resources/jobs/_{schema}.{entity}.job.yml` (note the leading underscore in the filename) referencing the strategy notebook from `.rtlh_dataworks/` and the SQL/JSON paths via `${workspace.file_path}`.

### Step 7 — Reconciliation test

`notebooks/test_{schema}_{entity}.ipynb` — row-count + `SUM(measure)` parity vs the legacy Synapse table for the last full month / day.

### Step 8 — Sandbox run

```
databricks bundle deploy --target sandbox-<you>
databricks bundle run _<schema>.<entity>.job --target sandbox-<you>
```

---

## 5. File naming cheat-sheet (authoritative — from MCP)

| Artefact | Path | Pattern |
|---|---|---|
| SQL transform | `config/sql/` | `{schema}.{table_name}.{LoadType}.sql` (LoadType ∈ `FullLoad`, `DeltaLoad`) |
| Config JSON | `config/json/` | `{schema}.{entity}.config.json` |
| Test notebook | `notebooks/` | `test_{schema}_{entity}_{specifier}.ipynb` |
| Job | `resources/jobs/` | `_{schema}.{entity}.job.yml` (note leading `_`) |
| Pipeline (DLT) | `resources/pipelines/` | `{schema}.{entity}.pipeline.yml` |
| Flyway versioned | `sql_deployment/V{version}/` | `V{version}_{YYYYMMDDHHmmss}__{description}.sql` |
| Flyway repeatable | `sql_deployment/Repeatable/` | `afterMigrate__{description}.sql` |

---

## 6. SQL conventions (authoritative — from MCP)

### 6.1 Formatting

- **Keywords**: UPPERCASE (`SELECT`, `FROM`, `LEFT OUTER JOIN`, `CASE`, `CAST`, `COALESCE`, `TO_UTC_TIMESTAMP`).
- **Aliases**: short lowercase (`f`, `s`, `a`, `iip`, `tum`).
- **Columns**: snake_case in target; sources keep their PascalCase as-is in joins.
- **Schema names** must come from `${env_*}` placeholders, never hardcoded.

Template:
```sql
-- {what this transform does}
SELECT
    <primary_key_hash>
,   <source_identifier>
,   <business_keys>
,   <measure_columns>
,   <dimension_keys>
,   <system_columns>
FROM <source_table> AS <alias>
<dimension_joins>
WHERE <filters>
```

### 6.2 Hash & business keys

- Primary key: `XXHASH64(<source_identifier_or_business_keys>) AS <table_name>_hk`
  - Simple: `XXHASH64(iip.SourceIdentifier) AS inventory_hk`
  - Composite: `XXHASH64(CONCAT(COALESCE(f.SourceIdentifier, '-1'), '|', COALESCE(tum.TimeUsage_BK, '-1'))) AS fixed_plant_time_usage_event_hk`
- FK columns are auto-generated in the **table DDL**, not in the SELECT:
  ```sql
  <entity>_hk BIGINT NOT NULL GENERATED ALWAYS AS (XXHASH64(<entity>_bk))
  ```
- BKs from a source system are **prefixed** with the source code: `CONCAT('RTIO|', a.FixedPlantAsset_BK)`, `CONCAT('RTA|', m.SourceIdentifier)`. Exceptions: site codes (`'RTA'`, `'RTIO'`, `'Gove'`), hardcoded defaults (`'-1'`).

### 6.3 Null handling

| BK type | Default |
|---|---|
| Site | `'RTA'` or `'RTIO'` |
| Everything else | `'-1'` |

```sql
COALESCE(s.SiteCode, 'RTA')                                AS site_bk
COALESCE(l.SourceIdentifier, '-1')                         AS location_bk
COALESCE(CONCAT('RTA|', m.SourceIdentifier), '-1')         AS material_bk
```

Use `LEFT OUTER JOIN` to avoid losing fact rows when a dim lookup fails.

### 6.4 System columns (every ODM table)

DDL:
```sql
sys_data_quality      STRING
sys_is_active         TINYINT     NOT NULL
sys_created_datetime  TIMESTAMP   NOT NULL DEFAULT current_timestamp()
sys_updated_datetime  TIMESTAMP   NOT NULL
sys_data_source       STRING      NOT NULL
```

In the SELECT:
```sql
NULL                                                        AS sys_data_quality,
CAST(1 AS TINYINT)                                          AS sys_is_active,
CURRENT_TIMESTAMP()                                         AS sys_updated_datetime,
'${env_ingestion_catalog}.<schema>${env_schema_suffix}.<source_table>' AS sys_data_source
-- sys_created_datetime is auto-defaulted on first insert; do not select it
```

### 6.5 Dimension join pattern

```sql
LEFT OUTER JOIN ${env_ingestion_catalog}.<schema>${env_schema_suffix}.<dim_table> AS <a>
    ON  <fact>.<X_SK> = <a>.<X_SK>
    AND (<a>.<X_SK> > 0 OR <a>.<X_SK> IS NULL)
```

The `> 0 OR IS NULL` filter excludes invalid system keys while preserving fact rows.

### 6.6 Datetime / timezone

- All datetime columns stored in **UTC** via `TO_UTC_TIMESTAMP(<local_ts>, '<tz>')`.
- Per region: `Australia/Darwin` (Gove), `Australia/Perth` (RTIO/Pilbara), `Australia/Brisbane` (Brisbane/Weipa). Conditional via `CASE WHEN s.RegionCode = 'Gove' THEN 'Australia/Darwin' ELSE 'Australia/Brisbane' END`.
- For local reporting columns: keep local — `CAST(<ts> AS DATE) AS local_date_bk`, `DATE_FORMAT(<ts>, 'HHmm') AS local_time_bk`.
- Source date SK → date: `TO_DATE(CAST(<date_sk> AS STRING), 'yyyyMMdd')`.

---

## 7. CDF patterns (for `DeltaLoad`)

### Pattern 1 — `TABLE_CHANGES()` against base table
```sql
WITH f AS (
    SELECT *
    ,   ROW_NUMBER() OVER (
            PARTITION BY <pk_cols>
            ORDER BY _commit_version DESC, ModifiedAt DESC
        ) AS rnk
    FROM TABLE_CHANGES('${env_ingestion_catalog}.<schema>${env_schema_suffix}.<table>', {CT_WATERMARK})
    WHERE _change_type IN ('insert', 'delete', 'update_postimage')
)
SELECT
    <transformations>,
    CAST(CASE f._change_type WHEN 'delete' THEN 1 ELSE 0 END AS TINYINT) AS IsDelete
FROM f
<dim_joins>
WHERE f.rnk = 1;
```

### Pattern 2 — Persisted `_cdf` table
```sql
WITH f AS (
    SELECT *
    ,   ROW_NUMBER() OVER (
            PARTITION BY <pk_cols>
            ORDER BY _commit_version DESC, ModifiedAt DESC
        ) AS rnk
    FROM ${env_ingestion_catalog}.<schema>${env_schema_suffix}.<table>_cdf AS f
    WHERE _commit_version >= {CT_WATERMARK}
)
SELECT
    <transformations>,
    CAST(CASE f._change_type WHEN 'delete' THEN 1 ELSE 0 END AS TINYINT) AS IsDelete
FROM f
<dim_joins>
WHERE f.rnk = 1;
```

Rules:
- Always emit `IsDelete` for delta loads.
- `{CT_WATERMARK}` is injected by the framework.
- `ROW_NUMBER` ordering: `_commit_version DESC` first, then a datetime column (e.g. `ModifiedAt DESC`) when the partition key isn't the same as the target `_hk` source.

---

## 8. Mapping legacy operational scripts to Dataworks/Flyway

| Legacy script | Replaced by |
|---|---|
| Stage SQL Objects.ps1, Update SQL Project.ps1 | DAB targets (`dev`/`test`/`prod`); single code base. |
| RemoveIdentityColumn.sql | n/a — `_hk` is a generated `XXHASH64` column. |
| Update Existing Deleted Dimension Records.sql | `IsDelete` flag from CDF + framework MERGE. |
| Load fact partition.sql, Reload Cycle.sql, Reload Event.sql, Reload HMEAssetActivityType.sql, Reload Rail Tag Time Usage Event.sql | `ODM Delta Load Using CDF`; full reload = re-run with `{CT_WATERMARK}=0`. |
| Change Distribution.sql, Partition permission.sql | Delta `OPTIMIZE … ZORDER BY`; UC grants in Flyway `Repeatable/`. |
| CRI check - with date.sql, CRI QAL.sql | Test notebooks under `notebooks/test_*.ipynb`. |
| SchemaCompare.scmp, schema compare.sql, Deploy debug.sql | `flyway info` / `flyway validate`; `databricks bundle validate`. |

---

## 9. Worked example — `dim_HMETimeUsageModel_ORDW`

Legacy: HLP_DEV.vw_dim_HMETimeUsageModel_ORDW.sql — reads `vw_cdc_HSTG_UDLORDW_PUBLIC_V2_DIM_TIMEUSAGE`, joins `mde.PrimaryTUM` for TUM0–TUM5 defaults, builds `SourceIdentifier = TIMEUSAGE_SK | TIMEUSAGE_BK`, COALESCEs `'NA'` defaults, sort keys via `fn_ExtractNumbers`.

**Decision**: dim-style entity from a CDF source → there are two valid takes:
- HLP staging fed via a **persisted `_cdf`** + downstream Full Merge → cleanest 1:1 with the legacy pattern.
- Direct ODM Full Merge over the base source → simpler, recommended unless you genuinely need the persisted history.

Below = the simpler version.

`config/sql/odm.hme_time_usage_model.FullLoad.sql`:
```sql
-- HME Time Usage Model dim from ORDW DIM_TIMEUSAGE + MDE PrimaryTUM defaults
SELECT
    XXHASH64(
        CONCAT(
            COALESCE(d.TIMEUSAGE_SK, '-1'), '|',
            COALESCE(d.TIMEUSAGE_BK, '-1')
        )
    )                                                                           AS hme_time_usage_model_hk
,   CONCAT('RTIO|', COALESCE(d.TIMEUSAGE_BK, '-1'))                             AS hme_time_usage_model_bk
,   COALESCE(tum.TUM0Code, 'CT')                                                AS tum0_code
,   COALESCE(tum.TUM0Name, 'Calendar Time')                                     AS tum0_name
,   COALESCE(CONCAT(tum.TUM0Name, ' (', tum.TUM0Code, ')'),
             'Calendar Time (CT)')                                              AS tum0_display_name
,   COALESCE(tum.TUM5Code, 'NA')                                                AS tum5_code
,   COALESCE(tum.TUM5Name, 'NA')                                                AS tum5_name
,   d.TUM6Code                                                                  AS tum6_code
,   REPLACE(d.TUM6DISPLAYNAME, CONCAT(' (', d.TUM6Code, ')'), '')               AS tum6_name
,   d.TUM6DISPLAYNAME                                                           AS tum6_display_name
,   CAST(REGEXP_EXTRACT(d.TUM6Code, '[0-9]+', 0) AS BIGINT)                     AS tum6_sort_key
,   d.TUM7Code                                                                  AS tum7_code
,   REPLACE(d.TUM7DISPLAYNAME, CONCAT(' (', d.TUM7Code, ')'), '')               AS tum7_name
,   d.TUM7DISPLAYNAME                                                           AS tum7_display_name
,   CAST(REGEXP_EXTRACT(d.TUM7Code, '[0-9]+', 0) AS BIGINT)                     AS tum7_sort_key
,   NULL                                                                        AS sys_data_quality
,   CAST(1 AS TINYINT)                                                          AS sys_is_active
,   CURRENT_TIMESTAMP()                                                         AS sys_updated_datetime
,   '${env_ingestion_rtio_catalog}.ordw${env_schema_suffix}.public_v2_dim_timeusage' AS sys_data_source
FROM ${env_ingestion_rtio_catalog}.ordw${env_schema_suffix}.public_v2_dim_timeusage AS d
LEFT OUTER JOIN ${env_ingestion_catalog}.midas${env_schema_suffix}.mde_primary_tum AS tum
    ON tum.TUM5Code = d.TUM5Code
WHERE d.TIMEUSAGE_SK > 0;
```

`sql_deployment/V001/V001_20260428100000__Create_Table_odm.hme_time_usage_model.sql`:
```sql
CREATE TABLE IF NOT EXISTS ${env_target_catalog}.odm${env_schema_suffix}.hme_time_usage_model (
    hme_time_usage_model_hk   BIGINT  NOT NULL GENERATED ALWAYS AS (XXHASH64(hme_time_usage_model_bk)),
    hme_time_usage_model_bk   STRING  NOT NULL,
    tum0_code                 STRING,
    tum0_name                 STRING,
    tum0_display_name         STRING,
    tum5_code                 STRING,
    tum5_name                 STRING,
    tum6_code                 STRING,
    tum6_name                 STRING,
    tum6_display_name         STRING,
    tum6_sort_key             BIGINT,
    tum7_code                 STRING,
    tum7_name                 STRING,
    tum7_display_name         STRING,
    tum7_sort_key             BIGINT,
    sys_data_quality          STRING,
    sys_is_active             TINYINT     NOT NULL,
    sys_created_datetime      TIMESTAMP   NOT NULL DEFAULT current_timestamp(),
    sys_updated_datetime      TIMESTAMP   NOT NULL,
    sys_data_source           STRING      NOT NULL
) USING DELTA;
```

`config/json/odm.hme_time_usage_model.config.json` (shape — exact keys per the strategy's schema):
```json
{
  "strategy": "ODMFullMerge",
  "target_table": "${env_target_catalog}.odm${env_schema_suffix}.hme_time_usage_model",
  "source_query": "${workspace.file_path}/config/sql/odm.hme_time_usage_model.FullLoad.sql",
  "business_key_columns": ["hme_time_usage_model_bk"],
  "hash_key_column": "hme_time_usage_model_hk"
}
```

`resources/jobs/_odm.hme_time_usage_model.job.yml` — task that runs the `ODM Full Merge` notebook from `.rtlh_dataworks/` with the JSON above as the parameter source.

## 10. Worked example — `fact_HMETimeUsageEvent_ORDW` (DeltaLoad CDF)

Legacy: HLP_DEV.vw_fact_HMETimeUsageEvent_ORDW.sql — reads `rep_hstg_fact_HMETimeUsageEvent_ORDW`, computes a `groupEvent` CTE with lead/lag windowing, joins back.

Decision: fact + CDF source → **ODM Delta Load Using CDF**, Pattern 2 (`_cdf` table) so window functions work cleanly.

`config/sql/odm.fixed_plant_time_usage_event.DeltaLoad.sql` (skeleton — drop legacy plumbing, port T-SQL idioms):
```sql
-- Fixed plant time usage event from ORDW (delta load via persisted CDF)
WITH f AS (
    SELECT
        *
    ,   LEAD(RecordEventStart) OVER (PARTITION BY HMEAsset_SK ORDER BY RecordEventStart) AS NextRecordStart
    ,   LAG(RecordEventFinish, 2) OVER (PARTITION BY HMEAsset_SK ORDER BY RecordEventStart) AS Lag2RecordFinish
    ,   ROW_NUMBER() OVER (
            PARTITION BY HMETimeUsageEvent_SK
            ORDER BY _commit_version DESC, ModifiedAt DESC
        ) AS rnk
    FROM ${env_ingestion_rtio_catalog}.ordw${env_schema_suffix}.public_v2_fact_fixedplanttimeusage_cdf AS f
    WHERE _commit_version >= {CT_WATERMARK}
)
SELECT
    XXHASH64(CONCAT('RTIO|', COALESCE(f.RecordEventSourceIdentifier, '-1'))) AS fixed_plant_time_usage_event_hk
,   CONCAT('RTIO|', COALESCE(f.RecordEventSourceIdentifier, '-1'))           AS fixed_plant_time_usage_event_bk
,   COALESCE(s.SiteCode, 'RTIO')                                             AS site_bk
,   COALESCE(CONCAT('RTIO|', a.FixedPlantAsset_BK), '-1')                    AS fixed_plant_asset_bk
,   TO_UTC_TIMESTAMP(f.RecordEventStart,  'Australia/Perth')                 AS record_event_start_utc
,   TO_UTC_TIMESTAMP(f.RecordEventFinish, 'Australia/Perth')                 AS record_event_finish_utc
,   CAST(f.RecordEventStart AS DATE)                                         AS local_date_bk
,   DATE_FORMAT(f.RecordEventStart, 'HHmm')                                  AS local_time_bk
,   f.cRecordEventTUM                                                        AS c_record_event_tum
,   f.sRecordEventTUM                                                        AS s_record_event_tum
,   CAST(CASE
            WHEN f.NextRecordStart IS NOT NULL
             AND 10 > ((UNIX_TIMESTAMP(f.RecordEventStart) - UNIX_TIMESTAMP(f.Lag2RecordFinish)) / 60)
            THEN 0 ELSE 1
        END AS TINYINT)                                                      AS c_group_event_tum
,   NULL                                                                     AS sys_data_quality
,   CAST(1 AS TINYINT)                                                       AS sys_is_active
,   CURRENT_TIMESTAMP()                                                      AS sys_updated_datetime
,   '${env_ingestion_rtio_catalog}.ordw${env_schema_suffix}.public_v2_fact_fixedplanttimeusage' AS sys_data_source
,   CAST(CASE f._change_type WHEN 'delete' THEN 1 ELSE 0 END AS TINYINT)     AS IsDelete
FROM f
LEFT OUTER JOIN ${env_ingestion_rtio_catalog}.ordw${env_schema_suffix}.public_v2_dim_fixedplantasset AS a
    ON  f.FixedPlantAsset_SK = a.FixedPlantAsset_SK
    AND (a.FixedPlantAsset_SK > 0 OR a.FixedPlantAsset_SK IS NULL)
LEFT OUTER JOIN ${env_ingestion_rtio_catalog}.ordw${env_schema_suffix}.public_v2_dim_site AS s
    ON  f.Site_SK = s.Site_SK
    AND (s.Site_SK > 0 OR s.Site_SK IS NULL)
WHERE f.rnk = 1;
```

(One `.DeltaLoad.sql` per source system; the ODM target table `odm.fixed_plant_time_usage_event` is the union destination across all per-source delta-load jobs.)

---

## 11. T-SQL → Spark SQL conversion cheat-sheet

| T-SQL | Spark SQL |
|---|---|
| `isnull(x, y)` | `COALESCE(x, y)` |
| `getdate()` | `CURRENT_TIMESTAMP()` |
| `datediff(second, a, b)` | `(UNIX_TIMESTAMP(b) - UNIX_TIMESTAMP(a))` |
| `datediff(minute, a, b)` | `((UNIX_TIMESTAMP(b) - UNIX_TIMESTAMP(a)) / 60)` |
| `cast(x as datetime2(7))` | `CAST(x AS TIMESTAMP)` (UTC after `TO_UTC_TIMESTAMP`) |
| `cast(x as nvarchar(N))` | `CAST(x AS STRING)` |
| `cast(x as bit)` | `CAST(x AS BOOLEAN)` |
| `[schema].[object]` | `` `schema`.`object` `` (only if reserved) |
| `+` for string concat | `CONCAT(a, b)` |
| `top N` | `LIMIT N` |
| `iif(c, a, b)` | `CASE WHEN c THEN a ELSE b END` |
| `try_cast` | `TRY_CAST` |
| `HLP_DEV.fn_ExtractNumbers(x)` | `CAST(REGEXP_EXTRACT(x, '[0-9]+', 0) AS BIGINT)` |
| Window `lead/lag/row_number` | identical syntax |
| Identity SK | `<entity>_hk BIGINT GENERATED ALWAYS AS (XXHASH64(<entity>_bk))` |

---

## 12. Per-file conversion checklist (copy into PR description)

```
Legacy file: DW/PRiME.DW/PRiME.DW.Db/1. DEV/<HLP_DEV|PRES_DEV>/<path>/<file>.sql
Target schema:           odm | odm_<site>
Target table (snake_case):
Strategy chosen:         [ ] ODM Full Merge   [ ] ODM Delta Load Using CDF
                         [ ] Persisted CDF    [ ] Custom (justify)
CDF pattern (DeltaLoad): [ ] TABLE_CHANGES()  [ ] _cdf table
LoadType:                [ ] FullLoad   [ ] DeltaLoad
Business keys (_bk):
Sources mapped:
  - HSTG_V.<x>  →  ${env_ingestion_*}.<schema>${env_schema_suffix}.<x>
  - DAL_V.<y>   →  ${env_ingestion_*}.<schema>${env_schema_suffix}.<y>
Conventions checks:
  [ ] hash key column XXHASH64(...) AS <table>_hk
  [ ] BKs prefixed with source ('RTIO|', 'RTA|') where applicable
  [ ] COALESCE defaults ('-1' / 'RTA' / 'RTIO')
  [ ] LEFT OUTER JOIN with (X_SK > 0 OR X_SK IS NULL)
  [ ] All datetimes via TO_UTC_TIMESTAMP(...) for UTC; local_date_bk / local_time_bk for local
  [ ] sys_data_quality / sys_is_active / sys_updated_datetime / sys_data_source populated
  [ ] env placeholders only — no hardcoded catalog/schema
  [ ] DeltaLoad: ROW_NUMBER PARTITION BY <pk> ORDER BY _commit_version DESC, <ModifiedAt> DESC; WHERE rnk = 1
  [ ] DeltaLoad: IsDelete column emitted
Legacy plumbing dropped:
  [ ] OMD_*  [ ] CDC_*  [ ] DataLineage_SK  [ ] IsStub  [ ] LoadExecutionId  [ ] InstanceId
T-SQL idioms ported:
  [ ] isnull→COALESCE  [ ] datediff  [ ] cast types  [ ] string concat  [ ] fn_ExtractNumbers
Files added/modified:
  - config/sql/<schema>.<entity>.<LoadType>.sql
  - config/json/<schema>.<entity>.config.json
  - sql_deployment/V{ver}/V{ver}_{ts}__Create_Table_<schema>.<entity>.sql
  - resources/jobs/_<schema>.<entity>.job.yml
  - notebooks/test_<schema>_<entity>.ipynb
Sandbox:                 [ ] deployed  [ ] succeeded  [ ] reconcile passed
```

---

## 13. Common pitfalls (MCP-aligned)

1. **Using `_sk` for the new hash key.** It's `_hk` in Dataworks. `_bk` is the business key string column; `_hk` is the auto-generated `XXHASH64(_bk)` `BIGINT`.
2. **Joining on `rtlh_row_hash`.** It changes when business data changes; always join on business keys.
3. **Hardcoded catalog / schema names.** Use `${env_ingestion_catalog}`, `${env_ingestion_rtio_catalog}`, `${env_target_catalog}`, `${env_schema_suffix}`.
4. **Forgetting `IsDelete` in delta loads.** Required by the strategy contract.
5. **Forgetting `ROW_NUMBER … WHERE rnk = 1` dedup** when multiple commits hit the same PK.
6. **Mixing local & UTC.** Storage = UTC via `TO_UTC_TIMESTAMP`. Local dims = `local_date_bk` / `local_time_bk`.
7. **Skipping `(SK > 0 OR SK IS NULL)` on dim joins.** Lets invalid system keys pollute the result.
8. **Unprefixed BKs.** `material_bk` from RTIO without the `'RTIO|'` prefix collides with RTA values.
9. **Hardcoding `'RTA'` everywhere as a default.** Default site = `'RTA'` or `'RTIO'`; everything else = `'-1'`.
10. **One giant `Custom` notebook.** Try Full Merge / Delta Load CDF first; only fall back to a custom ODW transformation when the strategy doesn't fit.
11. **Editing `.rtlh_dataworks/`.** Reserved. Project-local helpers go in `core/`.
12. **Wrong filename**: `<schema>.<entity>.<LoadType>.sql` for SQL, `_<schema>.<entity>.job.yml` (leading `_`) for jobs, `V{ver}_{YYYYMMDDHHmmss}__{desc}.sql` for migrations. Linting CI will reject otherwise.

# Doing Ingestion and Transformation with RTLH Dataworks (PRiME context)

A practical guide for the OGR team. The first half tells you **which reusable notebooks Dataworks gives you and when to use each one**. The second half is a **complete worked example** you can copy when you build your first transformation.

Everything technical here is taken from the Dataworks docs (via the Dataworks MCP). Plain-language explanations are mine.

---

## Part 1 — What Dataworks gives you

### 1.1 What "Dataworks" actually is, in one paragraph

Dataworks is RTLH's internal Python framework. It bundles three things together:
1. **Databricks Asset Bundles (DAB)** — to package and deploy jobs / notebooks / clusters per environment.
2. **Flyway** — to version-control table DDL (so schema changes are reviewable and reproducible across dev/test/prod).
3. **Reusable notebooks** — pre-built Databricks notebooks that already implement the common ingestion and transformation patterns. You don't write the orchestration code; you write a SQL query and a small JSON/YAML config, and the reusable notebook does the rest.

Anything you build in Dataworks goes into a folder layout the framework expects. The reusable notebooks live in a reserved folder called `.rtlh_dataworks/` — **don't edit that folder**, just call the notebooks from your job YAML.

### 1.2 Ingestion — usually NOT your problem

For OGR / PRiME, **the Data Integration team owns ingestion**. They land every source you need into Unity Catalog at:

```
_ogr_azr_syd.<source_schema>.<table>
```

Those tables are Delta-format with **Change Data Feed (CDF)** turned on, and they each carry two metadata columns: `rtlh_row_hash` (a hash of the business columns) and `rtlh_ingestion_time` (when the row was last loaded). You read from these and that's it — you don't build the pipelines that fill them.

**Only build your own ingestion if Data Integration genuinely cannot deliver the source.** If you have to, here is the menu of reusable Dataworks ingestion strategies (each one is a notebook you point at a SQL query + config). Plain-language descriptions:

| Strategy notebook | What it does, in plain terms | When to pick it |
|---|---|---|
| **Full Merge Ingestion** | Reads the entire source table every run and merges into the target. Simple and safe; handles schema changes broadly. | Small or medium tables that are cheap to scan in full — e.g. reference data, lookup tables, small dims. |
| **Delta Load Using Change Tracking** | Uses SQL Server's built-in Change Tracking feature to know what changed since the last run. First run does a full load automatically. | You're pulling from a SQL Server source that already has Change Tracking enabled. |
| **Delta Load Using Watermark Column** | Picks rows where some monotonically-increasing column (like `ModifiedAt`) is greater than the last value it saw. Doesn't notice deletes. | Source has a reliable watermark column and you only care about inserts and updates. |
| **Delta Load Using Watermark with All Keys** | Same as the watermark strategy, plus it pulls all current primary keys to detect rows that were hard-deleted from source. | Same as above but you also need to mirror deletes. |
| **Watermark Forward Only** | Append-only — every new row gets inserted; nothing gets merged. | Logs, telemetry, immutable event streams. |
| **Ranged Reload** | Drops and re-ingests a defined date window every run (e.g. last 6 months). | Sources that backfill late-arriving data — equivalent to PRiME's "load last 3 / 6 months for Cycle and Event" pattern. |
| **DLT Ingestion.CloudFile** | Auto-loader-based ingestion of files from ADLS, used inside a DLT pipeline. | Source is files (CSV/JSON/Parquet) sitting in cloud storage. |

Supported source drivers across these: SQL Server, Oracle, PIAF, Delta Catalog (Unity), and cloud files (DLT only). All of them auto-detect the first run (target empty → full load) and then switch to incremental.

For PRiME: the Data Integration team has effectively replaced the legacy "UDL Extractor" with this catalogue. You shouldn't need to touch any of it.

### 1.3 Transformation — what OGR actually builds

Transformation is the part you own. Dataworks gives you a small number of reusable transformation notebooks; you supply a SQL query (the actual transformation logic) and some config, and the framework runs it for you.

| Strategy notebook | What it does, in plain terms | When to pick it (PRiME mapping) |
|---|---|---|
| **ODM Full Merge** | Runs your `FullLoad` SQL, then merges the output into the ODM target table — inserting new rows, updating changed rows, deleting rows that disappeared. | **Default for dimensions and small reference tables.** Replaces the entire legacy `HLP_DEV.dim_*` + custom CDC stack. |
| **ODM Delta Load Using Change Data Feed** | Reads only what changed in the source by using its CDF (`TABLE_CHANGES('<table>', last_version)`), runs your `DeltaLoad` SQL on those rows, and merges into the target. Falls back to a full load on first run / empty target. | **Default for facts.** Every `_ogr_azr_syd.*` source has CDF on, so this is what we use. Replaces every `HLP_DEV.fact_*` + `rep_*` + `vw_cdc_*` pattern. |
| **Persisted Change Data Feed** | A DLT streaming notebook that copies a source's CDF history into a persistent `_cdf` table. You then read that `_cdf` table from a Delta Load notebook downstream. | Optional — only use if you (a) need to keep a long history of CDF events for audit, (b) have multiple consumers of the same change feed, or (c) need extra window functions (lead/lag) in the transformation that work better against a persisted source. |
| **Custom ODW Transformation Notebook** | A blank framework template where you write your own steps in Python or `%sql` cells. | Only when none of the above fit — for example, a multi-step pipeline that mixes Python and SQL, or that needs to write to several tables in one run. **Last resort.** |
| **DLT Transformation notebooks** (Quarantine, SCD, MaterializedView) | Standard DLT-based transformation building blocks that chain into a DLT pipeline. | Only if your data product is implemented as a DLT pipeline rather than a job. Not the default for OGR. |

### 1.4 How to pick (in plain English)

- **Is the legacy object a dimension or small reference table?** → ODM Full Merge.
- **Is it a fact (event-grain, large, mostly append)?** → ODM Delta Load Using CDF.
- **Does the legacy logic need multiple steps, mixed Python+SQL, or write multiple tables?** → Custom ODW Transformation Notebook.
- **Is the team committed to DLT for this product?** → DLT transformation notebooks.

For OGR / PRiME, in practice it's almost always one of the first two.

### 1.5 Mapping legacy PRiME pieces to Dataworks

| Legacy PRiME piece | What to do |
|---|---|
| `HLP_DEV.vw_dim_*` + `HLP_DEV.dim_*` (per-source dim with persisted SK) | Migrate as **ODM Full Merge** with a `.FullLoad.sql`. The `_hk` hash column replaces the persisted sequence SK. |
| `HLP_DEV.vw_fact_*` + `HLP_DEV.fact_*` (per-source fact) | Migrate as **ODM Delta Load Using CDF** with a `.DeltaLoad.sql`. |
| `HLP_DEV.vw_hstg_*` (multi-table flatten before dim/fact) | Don't make it a separate notebook — fold this SQL into the downstream `.FullLoad.sql` / `.DeltaLoad.sql`. |
| `HLP_DEV.rep_*`, `HLP_DEV.vw_cdc_*`, `HLP_DEV.cdc.*` | **Delete.** Source CDF replaces all of this. |
| `HLP_DEV.mde.*` | Migrate as **ODM Full Merge** reading from `_ogr_azr_syd.midas.*`. |
| `PRES_DEV.dim_*` (OCDM-conformed dim across sources) | An **ODM Full Merge** whose query `UNION ALL`s the per-source HLP outputs. |
| `PRES_DEV.fact_*` / `m_fact_*` | **ODM Delta Load Using CDF**, partitioned by event-month. |
| `PRES_<DATASET>_DEV.*` (gold / star schema) | Out of scope for this guide — handled in the gold-layer guide. |
| `DW/Scripts/*.sql` operational helpers | Replaced by framework features — Flyway for DDL, DAB targets for environments, framework merge logic for `IsDelete` / "deleted dimension records". |

### 1.6 Conventions you must follow in every transformation SQL

These are non-negotiable — Dataworks expects them, and CI/lint will reject them otherwise.

- **Hash key**: `XXHASH64(<source_id_or_business_keys>) AS <table>_hk`. The `_hk` column on the table itself is `GENERATED ALWAYS AS (XXHASH64(<entity>_bk))`, so you don't repeat the hash expression in multiple places.
- **Business keys** carry a source-system prefix so values from different sources can't collide: `CONCAT('RTIO|', <bk>)`, `CONCAT('RTA|', <bk>)`, etc.
- **Defaults for missing values**: site = `'RTA'` or `'RTIO'`; everything else = `'-1'`. Always wrap dim lookups in `COALESCE`.
- **Joins to dim tables**: `LEFT OUTER JOIN ... AND (alias.X_SK > 0 OR alias.X_SK IS NULL)`. The filter excludes invalid system keys but keeps the fact row when the dim lookup misses.
- **Datetimes**: store everything in UTC via `TO_UTC_TIMESTAMP(<local_ts>, '<tz>')`. Per region: `Australia/Darwin` (Gove), `Australia/Perth` (RTIO/Pilbara), `Australia/Brisbane` (Brisbane/Weipa). For local reporting, also expose `local_date_bk` (`CAST(... AS DATE)`) and `local_time_bk` (`DATE_FORMAT(..., 'HHmm')`).
- **System columns** on every ODM table: `sys_data_quality NULL`, `sys_is_active CAST(1 AS TINYINT)`, `sys_updated_datetime CURRENT_TIMESTAMP()`, `sys_data_source '<full source FQN>'`. (`sys_created_datetime` is a DDL default — don't select it in the transform.)
- **Delta loads** must dedup with `ROW_NUMBER() OVER (PARTITION BY <pk> ORDER BY _commit_version DESC, ModifiedAt DESC)` and filter `WHERE rnk = 1`. They must also emit an `IsDelete` column: `CAST(CASE _change_type WHEN 'delete' THEN 1 ELSE 0 END AS TINYINT) AS IsDelete`.
- **No hardcoded catalog or schema names.** Always use the framework placeholders: `${env_ingestion_catalog}`, `${env_ingestion_rtio_catalog}`, `${env_target_catalog}`, `${env_schema_suffix}`, and `${workspace.file_path}` for any file paths.

### 1.7 File and folder naming (also non-negotiable)

| What | Where it lives | How it's named |
|---|---|---|
| Transformation SQL | `config/sql/` | `{schema}.{table_name}.{FullLoad\|DeltaLoad}.sql` |
| Transformation config | `config/json/` | `{schema}.{entity}.config.json` |
| Test notebook | `notebooks/` | `test_{schema}_{entity}.ipynb` |
| Job definition | `resources/jobs/` | `_{schema}.{entity}.job.yml` (note the leading underscore) |
| DDL migration | `sql_deployment/V{version}/` | `V{version}_{YYYYMMDDHHmmss}__{description}.sql` |
| Repeatable DDL | `sql_deployment/Repeatable/` | `afterMigrate__{description}.sql` |

---

## Part 2 — Worked example: migrating `dim_HMETimeUsageModel` from PRiME

This walks one dim, end to end. The pattern is the same for any other entity — replace the names and source paths.

### 2.1 Find the legacy artefacts in the PRiME repo

| What | Where |
|---|---|
| Per-source dim transformation SQL | HLP_DEV.vw_dim_HMETimeUsageModel_ORDW.sql |
| Other source slices for the same dim | sibling files: `..._CM_KUC.sql`, `..._HSP.sql`, `..._MMRS_*.sql`, `..._ODP_Bauxite.sql` (one per source system) |
| Multi-table flatten feeding the dim | vw_hstg — fold this into your `.FullLoad.sql`; do **not** create a separate notebook |
| Persisted dim with stable SK | `HLP_DEV.dim_HMETimeUsageModel_*` — replaced by hash-keyed Dataworks table |
| OCDM-conformed dim | `PRES_DEV.dim_HMETimeUsageModel.sql` — becomes a downstream ODM Full Merge that `UNION ALL`s the per-source outputs |
| `rep_*` / `vw_cdc_*` / `cdc.*` | **Delete these.** Source CDF replaces them. |

**Strategy chosen**: this is a dim with a small, fully-scannable source set, so → **ODM Full Merge**.

### 2.2 Find the new source tables

| Legacy reference (Synapse) | New Databricks reference |
|---|---|
| `HSTG_V.VW_HSTG_UDLORDW_PUBLIC_V2_DIM_TIMEUSAGE` (and its `vw_cdc_*` view) | `${env_ingestion_rtio_catalog}.ordw${env_schema_suffix}.public_v2_dim_timeusage` |
| `HLP_DEV.vw_mde_PrimaryTUM` (MiDAS reference) | `${env_ingestion_catalog}.midas${env_schema_suffix}.mde_primary_tum` |

Confirm with Data Integration that both are present and CDF-enabled before starting.

### 2.3 Scaffold the project (only if it doesn't exist yet)

If the OGR HME data product folder doesn't exist:

```bash
CreateDataworks project --project-name ogr_hme \
    --project-config schema_name=odm target_catalog=${env_target_catalog} ...
```

This generates the standard Dataworks layout. Don't touch `.rtlh_dataworks/`.

### 2.4 Write the transformation SQL

Create `config/sql/odm.hme_time_usage_model_ordw.FullLoad.sql`:

```sql
-- HME Time Usage Model from RTIO ORDW DIM_TIMEUSAGE,
-- defaulted from MiDAS PrimaryTUM for the higher TUM levels.
-- Replaces legacy HLP_DEV.vw_dim_HMETimeUsageModel_ORDW.

SELECT
    XXHASH64(
        CONCAT(
            COALESCE(CAST(d.TIMEUSAGE_SK AS STRING), '-1'), '|',
            COALESCE(d.TIMEUSAGE_BK, '-1')
        )
    )                                                                       AS hme_time_usage_model_hk
,   CONCAT('RTIO|', COALESCE(d.TIMEUSAGE_BK, '-1'))                         AS hme_time_usage_model_bk
,   COALESCE(tum.TUM0Code, 'CT')                                            AS tum0_code
,   COALESCE(tum.TUM0Name, 'Calendar Time')                                 AS tum0_name
,   COALESCE(CONCAT(tum.TUM0Name, ' (', tum.TUM0Code, ')'),
             'Calendar Time (CT)')                                          AS tum0_display_name
,   COALESCE(tum.TUM5Code, 'NA')                                            AS tum5_code
,   COALESCE(tum.TUM5Name, 'NA')                                            AS tum5_name
,   d.TUM6Code                                                              AS tum6_code
,   REPLACE(d.TUM6DISPLAYNAME, CONCAT(' (', d.TUM6Code, ')'), '')           AS tum6_name
,   d.TUM6DISPLAYNAME                                                       AS tum6_display_name
,   CAST(REGEXP_EXTRACT(d.TUM6Code, '[0-9]+', 0) AS BIGINT)                 AS tum6_sort_key
,   d.TUM7Code                                                              AS tum7_code
,   REPLACE(d.TUM7DISPLAYNAME, CONCAT(' (', d.TUM7Code, ')'), '')           AS tum7_name
,   d.TUM7DISPLAYNAME                                                       AS tum7_display_name
,   CAST(REGEXP_EXTRACT(d.TUM7Code, '[0-9]+', 0) AS BIGINT)                 AS tum7_sort_key
,   NULL                                                                    AS sys_data_quality
,   CAST(1 AS TINYINT)                                                      AS sys_is_active
,   CURRENT_TIMESTAMP()                                                     AS sys_updated_datetime
,   '${env_ingestion_rtio_catalog}.ordw${env_schema_suffix}.public_v2_dim_timeusage'
                                                                            AS sys_data_source
FROM ${env_ingestion_rtio_catalog}.ordw${env_schema_suffix}.public_v2_dim_timeusage AS d
LEFT OUTER JOIN ${env_ingestion_catalog}.midas${env_schema_suffix}.mde_primary_tum AS tum
    ON tum.TUM5Code = d.TUM5Code
WHERE d.TIMEUSAGE_SK > 0;
```

What this does, in plain terms:
- Reads the time-usage dim from the RTIO ORDW source (which Data Integration has ingested from the on-prem RTIO data warehouse).
- Looks up the higher TUM levels (TUM0–TUM5) from a MiDAS-maintained reference table; if the lookup misses, it fills in defaults (`'NA'`, `'CT'`, etc.) so we never produce nulls in business columns.
- Builds a stable, source-prefixed business key (`'RTIO|...'`) and the matching hash key (`hme_time_usage_model_hk`), so OGR can later combine this slice with `'CM_KUC|...'`, `'HSP|...'`, etc., without collisions.
- Stamps the row with audit columns saying when it was loaded and where from.

Convention checklist (pulled straight from the Dataworks conventions):
- Hash key uses `XXHASH64(...)` and the column is named `<table>_hk`.
- BK is prefixed with the source code (`'RTIO|'`).
- Defaults applied with `COALESCE` (`'-1'`, `'NA'`, etc.).
- Lookup is a `LEFT OUTER JOIN` so source rows aren't lost.
- All four required system columns are populated.
- No hardcoded catalogs/schemas; all `${env_*}` placeholders.
- Legacy plumbing dropped: `DataLineage_SK`, `IsStub`, `InstanceId`, `CDC_KeyHash`, `CDC_RecordHash`, `CDC_DateTime`, `CDC_Action`.

Repeat the file for each source slice (one `.FullLoad.sql` per source system), each producing rows with its own source-prefixed `_bk`. The final OCDM dim is just the union of these.

### 2.5 Write the table DDL (Flyway migration)

Create `sql_deployment/V001/V001_20260428100000__Create_Table_odm.hme_time_usage_model.sql`:

```sql
CREATE TABLE IF NOT EXISTS ${env_target_catalog}.odm${env_schema_suffix}.hme_time_usage_model (
    hme_time_usage_model_hk   BIGINT      NOT NULL GENERATED ALWAYS AS (XXHASH64(hme_time_usage_model_bk)),
    hme_time_usage_model_bk   STRING      NOT NULL,
    tum0_code                 STRING,
    tum0_name                 STRING,
    tum0_display_name         STRING,
    tum5_code                 STRING,
    tum5_name                 STRING,
    tum6_code                 STRING,
    tum6_name                 STRING,
    tum6_display_name         STRING,
    tum6_sort_key             BIGINT,
    tum7_code                 STRING,
    tum7_name                 STRING,
    tum7_display_name         STRING,
    tum7_sort_key             BIGINT,
    sys_data_quality          STRING,
    sys_is_active             TINYINT     NOT NULL,
    sys_created_datetime      TIMESTAMP   NOT NULL DEFAULT current_timestamp(),
    sys_updated_datetime      TIMESTAMP   NOT NULL,
    sys_data_source           STRING      NOT NULL
) USING DELTA;
```

What this does, in plain terms:
- Creates the Delta table the Full Merge will write into.
- The `_hk` column is computed automatically by Delta from `_bk`, so it's always consistent and you never set it directly.
- Filename pattern is the Dataworks-required `V{version}_{YYYYMMDDHHmmss}__{description}.sql` so Flyway can version it across environments.

### 2.6 Wire up the job

Create `resources/jobs/_odm.hme_time_usage_model.job.yml` (note the leading underscore — that's the Dataworks convention):

```yaml
# ODM HME Time Usage Model — ODM Full Merge for ORDW source slice.
resources:
  jobs:
    odm_hme_time_usage_model_ordw:
      name: odm_hme_time_usage_model_ordw
      email_notifications:
        on_failure:
          - ${workspace.current_user.userName}

      job_clusters:
        - job_cluster_key: odp_vco_cluster
          new_cluster: ${var.env_job_cluster_32}

      tasks:
        - task_key: odm_hme_time_usage_model_ordw
          existing_cluster_id: ${var.env_cluster_id}
          # job_cluster_key: odp_vco_cluster
          notebook_task:
            notebook_path: ../../.rtlh_dataworks/RTLHDataworksCommon/notebooks/Transformation.ODMFullMerge.ipynb
            base_parameters:
              TargetCatalog:        ${var.env_target_catalog}
              TargetSchema:         odm
              EnvSchemaSuffix:      ${var.env_schema_suffix}
              TargetTable:          hme_time_usage_model
              FullLoadQueryPath:    ${workspace.file_path}/config/sql/odm.hme_time_usage_model_ordw.FullLoad.sql
              BusinessKeyColumns:   hme_time_usage_model_bk
              HashKeyColumn:        hme_time_usage_model_hk
              Verbose:              "false"
```

What this does, in plain terms:
- Defines a Databricks job that runs the **ODM Full Merge** reusable notebook from the reserved `.rtlh_dataworks/` folder.
- Tells the notebook where to read the SQL (`FullLoadQueryPath`), where to write (`TargetCatalog` / `TargetSchema` / `TargetTable`), and what the business / hash keys are. The notebook handles the rest (executing the SQL, comparing to existing rows, inserting / updating / deleting).
- The exact `base_parameters` keys depend on the version of the strategy notebook your project pins; check with `dataworks_learn` for that strategy if in doubt. The pattern is stable.
- If you migrate multiple source slices, add one task per slice under `tasks:`, all writing to the same `TargetTable`.

### 2.7 Add a reconciliation test

Create `notebooks/test_odm_hme_time_usage_model.ipynb`. The test must `dbutils.notebook.exit(...)` a JSON string with at least an `all_passed` boolean — the framework runner uses that to pass/fail. A useful shape:

```json
{
  "all_passed": true,
  "details": {
    "target":  { "catalog": "...", "schema": "odm", "table": "hme_time_usage_model", "row_count": 1234 },
    "legacy":  { "table": "PRES_DEV.dim_HMETimeUsageModel", "row_count": 1233 },
    "tolerance_pct": 1.0
  }
}
```

What this does, in plain terms:
- Compares the new Databricks table against the equivalent legacy Synapse table for the most recent full day.
- Checks row count and a few `COUNT(DISTINCT)` cardinalities (e.g. distinct `tum6_code`, `tum7_code`).
- Has to pass before we can cut over the GMR-defined Power BI measures from the legacy gold layer to the new one.

### 2.8 Deploy to your sandbox and run

```bash
databricks bundle deploy --target sandbox-<your-name>
databricks bundle run odm_hme_time_usage_model_ordw --target sandbox-<your-name>
```

Then verify in the sandbox:

```sql
SELECT COUNT(*),
       COUNT(DISTINCT hme_time_usage_model_bk),
       MIN(sys_updated_datetime), MAX(sys_updated_datetime)
FROM   <your_sandbox_catalog>.odm.hme_time_usage_model;
```

Iterate on the SQL until row counts and cardinalities line up with the legacy Synapse table, then PR the change.

### 2.9 Generalise to the other source slices

Repeat steps 2.4 + 2.6 (one new `.FullLoad.sql` and one new task in the job YAML) for each remaining source: `_CM_KUC`, `_HSP`, `_MMRS_Boron`, `_MMRS_Gove`, `_MMRS_IOC`, `_MMRS_KUC`, `_MMRS_OT`, `_MMRS_Weipa`, `_ODP_Bauxite`. Each one:

- Reads from its own `${env_ingestion_*}.<schema>${env_schema_suffix}.<source_table>`,
- Prefixes `_bk` with its own source code (`'CM_KUC|'`, `'HSP|'`, `'BORON|'`, etc.),
- Writes to the **same** `odm.hme_time_usage_model` target table.

Once all per-source jobs run, the table contains the conformed cross-source dim — there's no extra "OCDM dim" notebook needed because every row already carries a unique source-prefixed `_bk`.

### 2.10 Things you should NOT do (common pitfalls)

- ❌ Recreate `rep_*`, `vw_cdc_*`, `cdc.*`, `IsStub`, `DataLineage_SK`, `LoadExecutionId`, `InstanceId` — all the legacy CDC plumbing is replaced by Dataworks' framework + the source CDF.
- ❌ Use IDENTITY columns for the SK — the SK is `_hk`, generated as `XXHASH64(_bk)`.
- ❌ Hardcode catalog or schema names — always `${env_*}` placeholders.
- ❌ Omit the source-system prefix on `_bk` — keys from RTIO and RTA will collide otherwise.
- ❌ Edit anything inside `.rtlh_dataworks/` — that folder is reserved for the framework.
- ❌ Skip the reconciliation test — cut-over is gated on it.

---

## TL;DR you can give to your co-workers in one breath

> "Data Integration drops every source we need into `_ogr_azr_syd.<source>.*` as Delta tables with CDF on. Our job is to turn each legacy PRiME `HLP_DEV` / `PRES_DEV` SQL object into a small Dataworks asset: a `FullLoad.sql` (for dims) or `DeltaLoad.sql` (for facts) under `config/sql/`, a Flyway DDL under `sql_deployment/`, and a job YAML under `resources/jobs/` that calls one of two reusable notebooks — **ODM Full Merge** for dims, **ODM Delta Load Using CDF** for facts. We follow the Dataworks naming, hash-key, default-value, dim-join, timezone, and system-column conventions, and we add a reconciliation test against the legacy Synapse table before cutting over. Anything more complex than that is a custom ODW transformation notebook."